In [1]:
# ============================================
# STEP 1) TILING + NEGATIVE SAMPLING + EVENT-ID FILENAMES
# ============================================

!pip -q install segmentation_models_pytorch

from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, random
import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

ZIP_ROOT  = "/content/drive/MyDrive/burned_area"
BASE_ROOT = "/content/burned_area"
OUT       = "/content/burned_tiles"

os.makedirs(BASE_ROOT, exist_ok=True)
os.makedirs(f"{OUT}/x", exist_ok=True)
os.makedirs(f"{OUT}/y", exist_ok=True)

# --- unzip all parts ---
for fname in sorted(os.listdir(ZIP_ROOT)):
    if fname.endswith(".zip"):
        part_name = fname.replace(".zip", "")
        out_dir = os.path.join(BASE_ROOT, part_name)
        os.makedirs(out_dir, exist_ok=True)
        print(f"Açılıyor: {fname}")
        with zipfile.ZipFile(os.path.join(ZIP_ROOT, fname), "r") as z:
            z.extractall(out_dir)

TILE = 256

# NEGATIVE SAMPLING SETTINGS
KEEP_NEG_RATIO = 0.15   # 0.10-0.30 arası deneyebilirsin
NEG_PER_POS_CAP = 2.0   # negatifleri pozitiflere göre üstten kıs (örn 2.0 => en fazla 2x negatif)

rng = np.random.default_rng(42)

pos_count = 0
neg_count = 0

# --- iterate PART1..PART5 ---
for part in sorted(os.listdir(BASE_ROOT)):
    part_dir = os.path.join(BASE_ROOT, part)
    if not os.path.isdir(part_dir):
        continue

    # inside: real dataset folder
    subdirs = [d for d in os.listdir(part_dir) if os.path.isdir(os.path.join(part_dir, d))]
    if not subdirs:
        continue
    ROOT = os.path.join(part_dir, subdirs[0])
    print(f"\nİşleniyor: {ROOT}")

    # events
    for event in tqdm(sorted(os.listdir(ROOT))):
        evdir = os.path.join(ROOT, event)
        if not os.path.isdir(evdir):
            continue

        masks = [f for f in os.listdir(evdir) if "mask" in f.lower()]
        s2s = [
            f for f in os.listdir(evdir)
            if f.lower().startswith("sentinel2_")
            and f.lower().endswith(".tiff")
            and "cloud" not in f.lower()
        ]
        if not masks or not s2s:
            continue

        # mask read (assume same H,W as images)
        with rasterio.open(os.path.join(evdir, masks[0])) as msk_src:
            mask_arr = msk_src.read(1)

        for s2 in sorted(s2s):
            with rasterio.open(os.path.join(evdir, s2)) as src:
                if src.count < 3:
                    continue

                H, W = src.height, src.width

                for top in range(0, H - TILE + 1, TILE):
                    for left in range(0, W - TILE + 1, TILE):
                        w = Window(left, top, TILE, TILE)

                        img_patch = src.read([1, 2, 3], window=w).astype(np.float32)  # (3,H,W)
                        mask_patch = mask_arr[top:top+TILE, left:left+TILE]

                        is_pos = (mask_patch > 0).any()

                        # --- balance logic ---
                        if not is_pos:
                            # random downsample negatives
                            if rng.random() > KEEP_NEG_RATIO:
                                continue
                            # cap negatives vs positives
                            if pos_count > 0 and neg_count >= int(NEG_PER_POS_CAP * pos_count):
                                continue

                        # event id & spatial id in filename
                        base_id = f"{part}__{event}__{os.path.splitext(s2)[0]}__t{top}_l{left}"
                        np.save(f"{OUT}/x/{base_id}.npy", img_patch)
                        np.save(f"{OUT}/y/{base_id}.npy", mask_patch)

                        if is_pos:
                            pos_count += 1
                        else:
                            neg_count += 1

print(f"\nSaved patches => pos={pos_count}, neg={neg_count}, total={pos_count+neg_count}")
print("OUT:", OUT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.9 MB/s eta 0:00:00
Mounted at /content/drive
Açılıyor: Satellite_burned_area_dataset_part1.zip
Açılıyor: Satellite_burned_area_dataset_part2.zip
Açılıyor: Satellite_burned_area_dataset_part3.zip
Açılıyor: Satellite_burned_area_dataset_part4.zip
Açılıyor: Satellite_burned_area_dataset_part5.zip

İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part1/Satellite_burned_area_dataset_part1


  0%|          | 0/17 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
100%|██████████| 17/17 [00:15<00:00,  1.09it/s]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part2/Satellite_burned_area_dataset_part2


100%|██████████| 11/11 [00:17<00:00,  1.55s/it]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part3/Satellite_burned_area_dataset_part3


...


İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part4/Satellite_burned_area_dataset_part4


100%|██████████| 7/7 [00:16<00:00,  2.37s/it]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part5/Satellite_burned_area_dataset_part5


100%|██████████| 9/9 [00:07<00:00,  1.25it/s]


Saved patches => pos=2330, neg=544, total=2874
OUT: /content/burned_tiles


In [2]:
import os
import random
import numpy as np

TILES_DIR = "/content/burned_tiles"
x_dir = os.path.join(TILES_DIR, "x")
y_dir = os.path.join(TILES_DIR, "y")

ids = sorted([os.path.splitext(f)[0] for f in os.listdir(x_dir) if f.endswith(".npy")])

random.seed(42)
random.shuffle(ids)

n = len(ids)
train_ids = ids[:int(0.7*n)]
val_ids   = ids[int(0.7*n):int(0.85*n)]
test_ids  = ids[int(0.85*n):]

print("Train:", len(train_ids), "Val:", len(val_ids), "Test:", len(test_ids))


Train: 2011 Val: 431 Test: 432


In [3]:
import numpy as np, os
from tqdm import tqdm

def stats(ids, tiles_dir):
    pos = 0
    total = 0
    frac_list = []
    for id_ in tqdm(ids):
        y = np.load(os.path.join(tiles_dir, "y", f"{id_}.npy"))
        total += 1
        frac = (y > 0).mean()
        frac_list.append(frac)
        if frac > 0:
            pos += 1
    return {
        "total": total,
        "pos_patches": pos,
        "neg_patches": total - pos,
        "avg_burn_frac": float(np.mean(frac_list)),
        "median_burn_frac": float(np.median(frac_list)),
    }

In [4]:
# Split + stats hücrelerini (STEP2 + STEP3) çalıştır

print("TRAIN", stats(train_ids, OUT))
print("VAL  ", stats(val_ids, OUT))
print("TEST ", stats(test_ids, OUT))


100%|██████████| 2011/2011 [00:00<00:00, 3867.39it/s]


TRAIN {'total': 2011, 'pos_patches': 1628, 'neg_patches': 383, 'avg_burn_frac': 0.423407624338113, 'median_burn_frac': 0.266998291015625}


100%|██████████| 431/431 [00:00<00:00, 4009.91it/s]


VAL   {'total': 431, 'pos_patches': 358, 'neg_patches': 73, 'avg_burn_frac': 0.47491196634597954, 'median_burn_frac': 0.43804931640625}


100%|██████████| 432/432 [00:00<00:00, 3767.34it/s]

TEST  {'total': 432, 'pos_patches': 344, 'neg_patches': 88, 'avg_burn_frac': 0.42166434393988717, 'median_burn_frac': 0.22507476806640625}


In [5]:
import torch
from torch.utils.data import Dataset
import numpy as np
import os

class BurnedPatchDataset(Dataset):
    def __init__(self, tiles_dir, ids, normalize=True):
        self.tiles_dir = tiles_dir
        self.ids = ids
        self.normalize = normalize

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]

        x = np.load(os.path.join(self.tiles_dir, "x", f"{id_}.npy"))
        y = np.load(os.path.join(self.tiles_dir, "y", f"{id_}.npy"))

        # (H,W,C) gelirse düzelt
        if x.ndim == 3 and x.shape[-1] == 3:
            x = np.transpose(x, (2, 0, 1))

        x = x.astype(np.float32)
        if self.normalize:
            x = x / 255.0

        y = (y > 0).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


## **Dataset and DataLoader Definition**

A custom PyTorch `Dataset` class is implemented to load image–mask pairs from disk.
Each sample consists of:
- A normalized RGB image tensor of shape (3, 256, 256)
- A binary segmentation mask of shape (256, 256)

PyTorch `DataLoader`s are used to enable efficient mini-batch loading and shuffling during training.


In [6]:
from torch.utils.data import DataLoader
TILES_DIR = "/content/burned_tiles"
train_ds = BurnedPatchDataset(TILES_DIR, train_ids)
val_ds   = BurnedPatchDataset(TILES_DIR, val_ids)
test_ds  = BurnedPatchDataset(TILES_DIR, test_ids)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False)


## **Model Architecture**

The segmentation model is based on the U-Net architecture with a ResNet18 encoder pretrained on ImageNet.

**Architecture details:**
- Encoder: ResNet18 (pretrained)
- Decoder: U-Net decoder
- Input channels: 3 (RGB)
- Output: 1-channel binary segmentation mask

U-Net is well-suited for this task due to its ability to preserve spatial details through skip connections.


In [7]:
import segmentation_models_pytorch as smp
import torch

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

# **Training U-NET**

In [ ]:
# ============================================================
# CONTROLLED ABLATION PIPELINE + SAVE EVERYTHING (Drive)
# FIXED: AMP/scaler bug + robust saving paths
# ============================================================
import os, copy, random, math, json
import numpy as np
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm



# === CHANGE THIS: where to save runs in your Drive ===
RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_unet"   # <- istediğin klasör
os.makedirs(RUNS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_model(encoder_name="resnet18", encoder_weights="imagenet"):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,
    )

def make_loss(loss_cfg):
    name = loss_cfg["name"]

    if name == "bce":
        bce = torch.nn.BCEWithLogitsLoss()
        return lambda logits, y: bce(logits, y), "BCE"

    if name == "bce_dice":
        bce = torch.nn.BCEWithLogitsLoss()
        dice = smp.losses.DiceLoss(mode="binary")
        w_bce  = loss_cfg.get("w_bce", 1.0)
        w_dice = loss_cfg.get("w_dice", 1.0)
        return (lambda logits, y: w_bce*bce(logits, y) + w_dice*dice(logits, y),
                f"BCE+Dice({w_bce},{w_dice})")

    if name == "focal_dice":
        alpha = loss_cfg.get("alpha", 0.8)
        gamma = loss_cfg.get("gamma", 2.0)
        focal = smp.losses.FocalLoss(mode="binary", alpha=alpha, gamma=gamma)
        dice  = smp.losses.DiceLoss(mode="binary")
        w_focal = loss_cfg.get("w_focal", 1.0)
        w_dice  = loss_cfg.get("w_dice", 1.0)
        return (lambda logits, y: w_focal*focal(logits, y) + w_dice*dice(logits, y),
                f"Focal(a={alpha},g={gamma})+Dice")

    raise ValueError(f"Unknown loss: {name}")

@torch.no_grad()
def dice_iou_from_logits(logits, y, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()
    inter = (pred * y).sum(dim=(1,2,3))

    union_d = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3))
    dice = ((2*inter + eps) / (union_d + eps)).mean().item()

    union_i = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) - inter
    iou = ((inter + eps) / (union_i + eps)).mean().item()
    return dice, iou

def run_epoch(model, loader, loss_fn, optimizer=None, use_amp=False, scaler=None, grad_clip=None, desc=""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    tot_loss = 0.0
    tot_dice = 0.0
    tot_iou  = 0.0
    n = 0

    # eval'de no_grad kullan
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in tqdm(loader, desc=desc):
            x = x.to(device, non_blocking=True)
            y = y.unsqueeze(1).float().to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast(True):
                    logits = model(x)
                    loss = loss_fn(logits, y)
            else:
                logits = model(x)
                loss = loss_fn(logits, y)

            if is_train:
                if use_amp:
                    scaler.scale(loss).backward()
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optimizer.step()

            # metric (logits'ten)
            dice, iou = dice_iou_from_logits(logits, y)

            tot_loss += float(loss.item())
            tot_dice += dice
            tot_iou  += iou
            n += 1

    if n == 0:
        return math.nan, math.nan, math.nan
    return tot_loss/n, tot_dice/n, tot_iou/n

def save_csv(path, rows, header):
    with open(path, "w") as f:
        f.write(",".join(header) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(h,"")) for h in header) + "\n")

def train_one(cfg):
    set_seed(cfg.get("seed", 42))

    # ---- run folder ----
    run_dir = os.path.join(RUNS_DIR, cfg["name"])
    os.makedirs(run_dir, exist_ok=True)

    # save config
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    best_path = os.path.join(run_dir, "best.pth")
    last_path = os.path.join(run_dir, "last.pth")
    hist_path = os.path.join(run_dir, "history.csv")

    model = build_model(cfg["encoder"], cfg["weights"]).to(device)
    loss_fn, loss_pretty = make_loss(cfg["loss"])

    # optimizer
    opt = cfg["optimizer"]["name"]
    lr  = cfg["optimizer"]["lr"]
    wd  = cfg["optimizer"].get("weight_decay", 0.0)
    if opt == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif opt == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        raise ValueError(opt)

    # scheduler (optional)
    scheduler = None
    if cfg.get("scheduler", None):
        sc = cfg["scheduler"]
        if sc["name"] == "plateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode=sc.get("mode","min"),
                factor=sc.get("factor",0.5),
                patience=sc.get("patience",2),
                min_lr=sc.get("min_lr",1e-6),
            )
        elif sc["name"] == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=sc.get("T_max", cfg["epochs"])
            )
        else:
            raise ValueError(sc["name"])

    # AMP
    use_amp = bool(cfg.get("amp", True)) and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

    # freeze
    freeze_epochs = int(cfg.get("freeze_epochs", 0))
    if freeze_epochs > 0:
        for p in model.encoder.parameters():
            p.requires_grad = False

    # early stopping
    es = cfg.get("early_stopping", {"enabled": False})
    es_enabled = bool(es.get("enabled", False))
    monitor = es.get("monitor","val_dice")
    mode    = es.get("mode","max")
    patience = int(es.get("patience", 6))

    best = -1e9 if mode=="max" else 1e9
    best_epoch = -1
    bad = 0

    grad_clip = cfg.get("grad_clip", None)
    plateau_on = cfg.get("plateau_on","val_loss")

    history = []

    for epoch in range(1, cfg["epochs"]+1):
        # unfreeze
        if freeze_epochs > 0 and epoch == freeze_epochs + 1:
            for p in model.encoder.parameters():
                p.requires_grad = True
            if cfg.get("reset_opt_on_unfreeze", True):
                new_lr = cfg.get("lr_unfreeze", lr)
                if opt == "adam":
                    optimizer = torch.optim.Adam(model.parameters(), lr=new_lr, weight_decay=wd)
                else:
                    optimizer = torch.optim.AdamW(model.parameters(), lr=new_lr, weight_decay=wd)

                # rebuild scheduler if exists
                if cfg.get("scheduler", None):
                    sc = cfg["scheduler"]
                    if sc["name"] == "plateau":
                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, mode=sc.get("mode","min"),
                            factor=sc.get("factor",0.5),
                            patience=sc.get("patience",2),
                            min_lr=sc.get("min_lr",1e-6),
                        )
                    elif sc["name"] == "cosine":
                        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                            optimizer, T_max=sc.get("T_max", cfg["epochs"])
                        )

                scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

        tl, td, ti = run_epoch(
            model, train_loader, loss_fn,
            optimizer=optimizer,
            use_amp=use_amp, scaler=scaler,
            grad_clip=grad_clip,
            desc=f"{cfg['name']} | {epoch}/{cfg['epochs']} TRAIN"
        )

        vl, vd, vi = run_epoch(
            model, val_loader, loss_fn,
            optimizer=None,
            use_amp=use_amp, scaler=scaler,
            grad_clip=None,
            desc=f"{cfg['name']} | {epoch}/{cfg['epochs']} VAL"
        )

        # scheduler
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                metric = vl if plateau_on=="val_loss" else (1.0 - vd)
                scheduler.step(metric)
            else:
                scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        row = {
            "epoch": epoch,
            "lr": lr_now,
            "train_loss": tl,
            "train_dice": td,
            "train_iou": ti,
            "val_loss": vl,
            "val_dice": vd,
            "val_iou": vi,
        }
        history.append(row)

        print(f"[{cfg['name']}] ep{epoch:02d} | TL {tl:.4f} | VL {vl:.4f} | VD {vd:.4f} | VI {vi:.4f} | LR {lr_now:.2e}")

        # always save LAST
        torch.save(
            {"epoch": epoch, "model": model.state_dict(), "cfg": cfg, "history": history},
            last_path
        )

        # best by monitor
        current = vd if monitor=="val_dice" else vl
        improved = (current > best) if mode=="max" else (current < best)

        if improved:
            best = current
            best_epoch = epoch
            bad = 0
            torch.save(
                {"epoch": epoch, "model": model.state_dict(), "cfg": cfg, "best": best, "history": history},
                best_path
            )
            print(f"  ✓ best saved ({monitor}={best:.6f}) -> {best_path}")
        else:
            if es_enabled:
                bad += 1
                if bad >= patience:
                    print(f"⛔ early stop (best ep={best_epoch}, best {monitor}={best:.6f})")
                    break

    # save history.csv
    header = ["epoch","lr","train_loss","train_dice","train_iou","val_loss","val_dice","val_iou"]
    save_csv(hist_path, history, header)

    # TEST with BEST (fallback to LAST if best missing)
    ckpt_path = best_path if os.path.exists(best_path) else last_path
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    test_loss, test_dice, test_iou = run_epoch(
        model, test_loader, loss_fn,
        optimizer=None,
        use_amp=use_amp, scaler=scaler,
        desc=f"{cfg['name']} | TEST ({'best' if ckpt_path==best_path else 'last'})"
    )

    summary = {
        "name": cfg["name"],
        "run_dir": run_dir,
        "best_path": best_path,
        "last_path": last_path,
        "hist_path": hist_path,
        "best_epoch": ckpt.get("epoch", None),
        "best_metric": ckpt.get("best", None),
        "test_loss": test_loss,
        "test_dice": test_dice,
        "test_iou": test_iou,
    }
    return summary

# ============================================================
# CONDITIONS
# ============================================================
BASE = {
    "name": "C00_BASE",
    "seed": 42,
    "encoder": "resnet18",
    "weights": "imagenet",
    "epochs": 30,
    "loss": {"name": "bce_dice", "w_bce": 1.0, "w_dice": 1.0},
    "optimizer": {"name": "adamw", "lr": 1e-4, "weight_decay": 1e-4},
    "scheduler": None,
    "plateau_on": "val_loss",
    "amp": False,          # True istersen otomatik scaler/amp çalışır
    "grad_clip": None,
    "freeze_epochs": 0,
    "reset_opt_on_unfreeze": True,
    "lr_unfreeze": 1e-4,
    "early_stopping": {"enabled": False, "monitor":"val_dice", "mode":"max", "patience":6},
}

ABLATIONS = [
    ("loss", {"name": "bce"}, "loss=BCE"),
    ("loss", {"name": "focal_dice", "alpha":0.8, "gamma":2.0}, "loss=Focal+Dice"),
    ("scheduler", {"name":"plateau","mode":"min","factor":0.5,"patience":2,"min_lr":1e-6}, "sched=plateau"),
    ("amp", True, "amp=on"),
    ("grad_clip", 1.0, "clip=1.0"),
    ("freeze_epochs", 2, "freeze=2ep"),
    ("early_stopping", {"enabled": True, "monitor":"val_dice", "mode":"max", "patience":6}, "ES=on"),
    ("optimizer.lr", 5e-5, "lr=5e-5"),
    ("optimizer.lr", 2e-4, "lr=2e-4"),
]

def apply_change(cfg, path, value):
    keys = path.split(".")
    ref = cfg
    for k in keys[:-1]:
        ref = ref[k]
    ref[keys[-1]] = value

def build_conditions(base_cfg, ablations):
    conds = [copy.deepcopy(base_cfg)]
    for i, (path, val, tag) in enumerate(ablations, start=1):
        c = copy.deepcopy(base_cfg)
        c["name"] = f"C{i:02d}_{tag}"
        apply_change(c, path, val)
        conds.append(c)
    return conds

CONDITIONS = build_conditions(BASE, ABLATIONS)

# ============================================================
# RUN ALL
# ============================================================
summaries = []
for cfg in CONDITIONS:
    summaries.append(train_one(cfg))

print("\n\nSaved runs under:", RUNS_DIR)
for s in summaries:
    print(s["name"], "->", s["run_dir"])
    print("  best:", s["best_path"])
    print("  last:", s["last_path"])
    print("  hist:", s["hist_path"])
    print("  best_epoch:", s["best_epoch"], "| test_dice:", f"{s['test_dice']:.4f}", "| test_iou:", f"{s['test_iou']:.4f}")


C00_BASE | 1/30 TRAIN: 100%|██████████| 252/252 [00:08<00:00, 28.34it/s]
C00_BASE | 1/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 67.77it/s]


[C00_BASE] ep01 | TL 1.0161 | VL 0.8118 | VD 0.4622 | VI 0.4125 | LR 1.00e-04
  ✓ best saved (val_dice=0.462162) -> /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE/best.pth


C00_BASE | 2/30 TRAIN: 100%|██████████| 252/252 [00:06<00:00, 36.44it/s]
C00_BASE | 2/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 74.40it/s]


[C00_BASE] ep02 | TL 0.8206 | VL 0.7362 | VD 0.5006 | VI 0.4490 | LR 1.00e-04
  ✓ best saved (val_dice=0.500566) -> /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE/best.pth


C00_BASE | 3/30 TRAIN: 100%|██████████| 252/252 [00:06<00:00, 36.20it/s]
C00_BASE | 3/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 71.22it/s]


...

[C09_lr=2e-4] ep29 | TL 0.1161 | VL 0.5532 | VD 0.6928 | VI 0.6515 | LR 2.00e-04


C09_lr=2e-4 | 30/30 TRAIN: 100%|██████████| 252/252 [00:07<00:00, 35.98it/s]
C09_lr=2e-4 | 30/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 71.82it/s]


[C09_lr=2e-4] ep30 | TL 0.1306 | VL 0.6449 | VD 0.6756 | VI 0.6279 | LR 2.00e-04


C09_lr=2e-4 | TEST (best): 100%|██████████| 54/54 [00:00<00:00, 67.64it/s]



Saved runs under: /content/drive/MyDrive/burned_area_outputs_unet
C00_BASE -> /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE
  best: /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE/best.pth
  last: /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE/last.pth
  hist: /content/drive/MyDrive/burned_area_outputs_unet/C00_BASE/history.csv
  best_epoch: 27 | test_dice: 0.6421 | test_iou: 0.6005
C01_loss=BCE -> /content/drive/MyDrive/burned_area_outputs_unet/C01_loss=BCE
  best: /content/drive/MyDrive/burned_area_outputs_unet/C01_loss=BCE/best.pth
  last: /content/drive/MyDrive/burned_area_outputs_unet/C01_loss=BCE/last.pth
  hist: /content/drive/MyDrive/burned_area_outputs_unet/C01_loss=BCE/history.csv
  best_epoch: 22 | test_dice: 0.6297 | test_iou: 0.5899
C02_loss=Focal+Dice -> /content/drive/MyDrive/burned_area_outputs_unet/C02_loss=Focal+Dice
  best: /content/drive/MyDrive/burned_area_outputs_unet/C02_loss=Focal+Dice/best.pth
  last: /content/drive/MyDrive/burne

In [ ]:
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

# mümkünse dataset uzunlukları
print("train n:", len(train_loader.dataset))
print("val n:", len(val_loader.dataset))
print("test n:", len(test_loader.dataset))


train batches: 252
val batches: 54
test batches: 54
train n: 2011
val n: 431
test n: 432


In [ ]:
train_ids = set(getattr(train_loader.dataset, "files", []))
val_ids   = set(getattr(val_loader.dataset, "files", []))
test_ids  = set(getattr(test_loader.dataset, "files", []))

print("train∩val:", len(train_ids & val_ids))
print("train∩test:", len(train_ids & test_ids))
print("val∩test:", len(val_ids & test_ids))


train∩val: 0
train∩test: 0
val∩test: 0


In [ ]:
x, y = next(iter(train_loader))
print("y min/max:", y.min().item(), y.max().item())
print("unique sample:", torch.unique(y[:1]).cpu()[:10])


y min/max: 0 1
unique sample: tensor([0, 1])


In [ ]:
import torch

def empty_mask_stats(loader, name=""):
    empty = 0
    total = 0
    pos_pixels = 0
    all_pixels = 0

    for x, y in loader:
        # y: (B,H,W) varsayıyorum
        y = y.float()
        # boş mask: tüm pikseller 0
        is_empty = (y.sum(dim=(1,2)) == 0)
        empty += is_empty.sum().item()
        total += y.size(0)

        pos_pixels += y.sum().item()
        all_pixels += y.numel()

    print(f"{name} empty masks: {empty}/{total} = {empty/total:.3f}")
    print(f"{name} foreground pixel ratio: {pos_pixels/all_pixels:.6f}")

empty_mask_stats(val_loader, "VAL")
empty_mask_stats(test_loader, "TEST")


VAL empty masks: 73/431 = 0.169
VAL foreground pixel ratio: 0.474912
TEST empty masks: 88/432 = 0.204
TEST foreground pixel ratio: 0.421664


In [ ]:
import torch

@torch.no_grad()
def baseline_all_zero(loader, thr=0.5, name=""):
    eps=1e-7
    dices=[]
    ious=[]
    for x,y in loader:
        y = y.unsqueeze(1).float().to(device)
        pred = torch.zeros_like(y)  # her şeyi background tahmin et
        inter = (pred*y).sum(dim=(1,2,3))

        union_d = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3))
        dice = ((2*inter + eps) / (union_d + eps)).mean().item()

        union_i = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) - inter
        iou = ((inter + eps) / (union_i + eps)).mean().item()

        dices.append(dice); ious.append(iou)

    print(f"{name} ALL-ZERO baseline | Dice={sum(dices)/len(dices):.4f} IoU={sum(ious)/len(ious):.4f}")

baseline_all_zero(val_loader, name="VAL")
baseline_all_zero(test_loader, name="TEST")


VAL ALL-ZERO baseline | Dice=0.1690 IoU=0.1690
TEST ALL-ZERO baseline | Dice=0.2037 IoU=0.2037


# **Training DeepLabv3**

In [ ]:
# ============================================================
# DeepLabV3+ CONTROLLED ABLATION PIPELINE + SAVE EVERYTHING (Drive)
# (same structure as your U-Net code, just model changed)
# ============================================================
import os, copy, random, math, json
import numpy as np
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

# === CHANGE THIS: where to save runs in your Drive ===
RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_deeplabv3"
os.makedirs(RUNS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ----------------------------
# Utils
# ----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------------------
# Model: DeepLabV3+
# ----------------------------
def build_model(encoder_name="resnet18", encoder_weights="imagenet"):
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,
        activation=None
    )


# ----------------------------
# Loss
# ----------------------------
def make_loss(loss_cfg):
    name = loss_cfg["name"]

    if name == "bce":
        bce = torch.nn.BCEWithLogitsLoss()
        return lambda logits, y: bce(logits, y), "BCE"

    if name == "bce_dice":
        bce = torch.nn.BCEWithLogitsLoss()
        dice = smp.losses.DiceLoss(mode="binary")
        w_bce  = loss_cfg.get("w_bce", 1.0)
        w_dice = loss_cfg.get("w_dice", 1.0)
        return (lambda logits, y: w_bce*bce(logits, y) + w_dice*dice(logits, y),
                f"BCE+Dice({w_bce},{w_dice})")

    if name == "focal_dice":
        alpha = loss_cfg.get("alpha", 0.8)
        gamma = loss_cfg.get("gamma", 2.0)
        focal = smp.losses.FocalLoss(mode="binary", alpha=alpha, gamma=gamma)
        dice  = smp.losses.DiceLoss(mode="binary")
        w_focal = loss_cfg.get("w_focal", 1.0)
        w_dice  = loss_cfg.get("w_dice", 1.0)
        return (lambda logits, y: w_focal*focal(logits, y) + w_dice*dice(logits, y),
                f"Focal(a={alpha},g={gamma})+Dice")

    raise ValueError(f"Unknown loss: {name}")


@torch.no_grad()
def dice_iou_from_logits(logits, y, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()

    inter = (pred * y).sum(dim=(1,2,3))

    union_d = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3))
    dice = ((2*inter + eps) / (union_d + eps)).mean().item()

    union_i = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) - inter
    iou = ((inter + eps) / (union_i + eps)).mean().item()

    return dice, iou


def run_epoch(model, loader, loss_fn, optimizer=None, use_amp=False, scaler=None, grad_clip=None, desc=""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    tot_loss = 0.0
    tot_dice = 0.0
    tot_iou  = 0.0
    n = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in tqdm(loader, desc=desc):
            x = x.to(device, non_blocking=True)
            y = y.unsqueeze(1).float().to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast(True):
                    logits = model(x)
                    loss = loss_fn(logits, y)
            else:
                logits = model(x)
                loss = loss_fn(logits, y)

            if is_train:
                if use_amp:
                    scaler.scale(loss).backward()
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optimizer.step()

            dice, iou = dice_iou_from_logits(logits, y)

            tot_loss += float(loss.item())
            tot_dice += dice
            tot_iou  += iou
            n += 1

    if n == 0:
        return math.nan, math.nan, math.nan
    return tot_loss/n, tot_dice/n, tot_iou/n


def save_csv(path, rows, header):
    with open(path, "w") as f:
        f.write(",".join(header) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(h,"")) for h in header) + "\n")


# ----------------------------
# Train one condition
# ----------------------------
def train_one(cfg):
    set_seed(cfg.get("seed", 42))

    run_dir = os.path.join(RUNS_DIR, cfg["name"])
    os.makedirs(run_dir, exist_ok=True)

    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    best_path = os.path.join(run_dir, "best.pth")
    last_path = os.path.join(run_dir, "last.pth")
    hist_path = os.path.join(run_dir, "history.csv")

    model = build_model(cfg["encoder"], cfg["weights"]).to(device)
    loss_fn, _ = make_loss(cfg["loss"])

    opt = cfg["optimizer"]["name"]
    lr  = cfg["optimizer"]["lr"]
    wd  = cfg["optimizer"].get("weight_decay", 0.0)
    if opt == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif opt == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        raise ValueError(opt)

    scheduler = None
    if cfg.get("scheduler", None):
        sc = cfg["scheduler"]
        if sc["name"] == "plateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode=sc.get("mode","min"),
                factor=sc.get("factor",0.5),
                patience=sc.get("patience",2),
                min_lr=sc.get("min_lr",1e-6),
            )
        elif sc["name"] == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=sc.get("T_max", cfg["epochs"])
            )
        else:
            raise ValueError(sc["name"])

    use_amp = bool(cfg.get("amp", True)) and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

    freeze_epochs = int(cfg.get("freeze_epochs", 0))
    if freeze_epochs > 0:
        for p in model.encoder.parameters():
            p.requires_grad = False

    es = cfg.get("early_stopping", {"enabled": False})
    es_enabled = bool(es.get("enabled", False))
    monitor = es.get("monitor","val_dice")  # or val_loss
    mode    = es.get("mode","max")          # max for dice, min for loss
    patience = int(es.get("patience", 6))

    best = -1e9 if mode=="max" else 1e9
    best_epoch = -1
    bad = 0

    grad_clip = cfg.get("grad_clip", None)
    plateau_on = cfg.get("plateau_on","val_loss")

    history = []

    for epoch in range(1, cfg["epochs"]+1):
        if freeze_epochs > 0 and epoch == freeze_epochs + 1:
            for p in model.encoder.parameters():
                p.requires_grad = True

            if cfg.get("reset_opt_on_unfreeze", True):
                new_lr = cfg.get("lr_unfreeze", lr)
                if opt == "adam":
                    optimizer = torch.optim.Adam(model.parameters(), lr=new_lr, weight_decay=wd)
                else:
                    optimizer = torch.optim.AdamW(model.parameters(), lr=new_lr, weight_decay=wd)

                if cfg.get("scheduler", None):
                    sc = cfg["scheduler"]
                    if sc["name"] == "plateau":
                        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, mode=sc.get("mode","min"),
                            factor=sc.get("factor",0.5),
                            patience=sc.get("patience",2),
                            min_lr=sc.get("min_lr",1e-6),
                        )
                    elif sc["name"] == "cosine":
                        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                            optimizer, T_max=sc.get("T_max", cfg["epochs"])
                        )

                scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

        tl, td, ti = run_epoch(
            model, train_loader, loss_fn,
            optimizer=optimizer,
            use_amp=use_amp, scaler=scaler,
            grad_clip=grad_clip,
            desc=f"{cfg['name']} | {epoch}/{cfg['epochs']} TRAIN"
        )

        vl, vd, vi = run_epoch(
            model, val_loader, loss_fn,
            optimizer=None,
            use_amp=use_amp, scaler=scaler,
            desc=f"{cfg['name']} | {epoch}/{cfg['epochs']} VAL"
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                metric = vl if plateau_on=="val_loss" else (1.0 - vd)
                scheduler.step(metric)
            else:
                scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        row = {
            "epoch": epoch,
            "lr": lr_now,
            "train_loss": tl,
            "train_dice": td,
            "train_iou": ti,
            "val_loss": vl,
            "val_dice": vd,
            "val_iou": vi,
        }
        history.append(row)

        print(f"[{cfg['name']}] ep{epoch:02d} | TL {tl:.4f} | VL {vl:.4f} | VD {vd:.4f} | VI {vi:.4f} | LR {lr_now:.2e}")

        torch.save(
            {"epoch": epoch, "model": model.state_dict(), "cfg": cfg, "history": history},
            last_path
        )

        current = vd if monitor=="val_dice" else vl
        improved = (current > best) if mode=="max" else (current < best)

        if improved:
            best = current
            best_epoch = epoch
            bad = 0
            torch.save(
                {"epoch": epoch, "model": model.state_dict(), "cfg": cfg, "best": best, "history": history},
                best_path
            )
            print(f"  ✓ best saved ({monitor}={best:.6f}) -> {best_path}")
        else:
            if es_enabled:
                bad += 1
                if bad >= patience:
                    print(f"⛔ early stop (best ep={best_epoch}, best {monitor}={best:.6f})")
                    break

    header = ["epoch","lr","train_loss","train_dice","train_iou","val_loss","val_dice","val_iou"]
    save_csv(hist_path, history, header)

    ckpt_path = best_path if os.path.exists(best_path) else last_path
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    test_loss, test_dice, test_iou = run_epoch(
        model, test_loader, loss_fn,
        optimizer=None,
        use_amp=use_amp, scaler=scaler,
        desc=f"{cfg['name']} | TEST ({'best' if ckpt_path==best_path else 'last'})"
    )

    summary = {
        "name": cfg["name"],
        "run_dir": run_dir,
        "best_path": best_path,
        "last_path": last_path,
        "hist_path": hist_path,
        "best_epoch": ckpt.get("epoch", None),
        "best_metric": ckpt.get("best", None),
        "test_loss": test_loss,
        "test_dice": test_dice,
        "test_iou": test_iou,
    }
    return summary


# ----------------------------
# Conditions (same idea)
# ----------------------------
BASE = {
    "name": "C00_BASE",
    "seed": 42,
    "encoder": "resnet18",
    "weights": "imagenet",
    "epochs": 30,
    "loss": {"name": "bce_dice", "w_bce": 1.0, "w_dice": 1.0},
    "optimizer": {"name": "adamw", "lr": 1e-4, "weight_decay": 1e-4},
    "scheduler": None,
    "plateau_on": "val_loss",
    "amp": False,
    "grad_clip": None,
    "freeze_epochs": 0,
    "reset_opt_on_unfreeze": True,
    "lr_unfreeze": 1e-4,
    "early_stopping": {"enabled": False, "monitor":"val_dice", "mode":"max", "patience":6},
}

ABLATIONS = [
    ("loss", {"name": "bce"}, "loss=BCE"),
    ("loss", {"name": "focal_dice", "alpha":0.8, "gamma":2.0}, "loss=Focal+Dice"),
    ("scheduler", {"name":"plateau","mode":"min","factor":0.5,"patience":2,"min_lr":1e-6}, "sched=plateau"),
    ("amp", True, "amp=on"),
    ("grad_clip", 1.0, "clip=1.0"),
    ("freeze_epochs", 2, "freeze=2ep"),
    ("early_stopping", {"enabled": True, "monitor":"val_dice", "mode":"max", "patience":6}, "ES=on"),
    ("optimizer.lr", 5e-5, "lr=5e-5"),
    ("optimizer.lr", 2e-4, "lr=2e-4"),
]

def apply_change(cfg, path, value):
    keys = path.split(".")
    ref = cfg
    for k in keys[:-1]:
        ref = ref[k]
    ref[keys[-1]] = value

def build_conditions(base_cfg, ablations):
    conds = [copy.deepcopy(base_cfg)]
    for i, (path, val, tag) in enumerate(ablations, start=1):
        c = copy.deepcopy(base_cfg)
        c["name"] = f"C{i:02d}_{tag}"
        apply_change(c, path, val)
        conds.append(c)
    return conds

CONDITIONS_deeplab = build_conditions(BASE, ABLATIONS)


# ----------------------------
# RUN ALL
# ----------------------------
summaries = []
for cfg in CONDITIONS_deeplab:
    summaries.append(train_one(cfg))

print("\n\nSaved runs under:", RUNS_DIR)
for s in summaries:
    print(s["name"], "->", s["run_dir"])
    print("  best:", s["best_path"])
    print("  last:", s["last_path"])
    print("  hist:", s["hist_path"])
    print("  best_epoch:", s["best_epoch"], "| test_dice:", f"{s['test_dice']:.4f}", "| test_iou:", f"{s['test_iou']:.4f}")


C00_BASE | 1/30 TRAIN: 100%|██████████| 252/252 [00:06<00:00, 41.22it/s]
C00_BASE | 1/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 70.36it/s]


[C00_BASE] ep01 | TL 0.9512 | VL 1.1055 | VD 0.5709 | VI 0.5183 | LR 1.00e-04
  ✓ best saved (val_dice=0.570946) -> /content/drive/MyDrive/burned_area_outputs_deeplabv3/C00_BASE/best.pth


C00_BASE | 2/30 TRAIN: 100%|██████████| 252/252 [00:05<00:00, 43.68it/s]
C00_BASE | 2/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 74.29it/s]


[C00_BASE] ep02 | TL 0.7811 | VL 0.9346 | VD 0.5035 | VI 0.4663 | LR 1.00e-04


C00_BASE | 3/30 TRAIN: 100%|██████████| 252/252 [00:06<00:00, 41.41it/s]
C00_BASE | 3/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 73.80it/s]


...

[C09_lr=2e-4] ep29 | TL 0.1719 | VL 0.5616 | VD 0.6571 | VI 0.6208 | LR 2.00e-04


C09_lr=2e-4 | 30/30 TRAIN: 100%|██████████| 252/252 [00:05<00:00, 42.92it/s]
C09_lr=2e-4 | 30/30 VAL: 100%|██████████| 54/54 [00:00<00:00, 72.54it/s]


[C09_lr=2e-4] ep30 | TL 0.1529 | VL 0.6579 | VD 0.6333 | VI 0.5934 | LR 2.00e-04


C09_lr=2e-4 | TEST (best): 100%|██████████| 54/54 [00:00<00:00, 69.73it/s]



Saved runs under: /content/drive/MyDrive/burned_area_outputs_deeplabv3
C00_BASE -> /content/drive/MyDrive/burned_area_outputs_deeplabv3/C00_BASE
  best: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C00_BASE/best.pth
  last: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C00_BASE/last.pth
  hist: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C00_BASE/history.csv
  best_epoch: 30 | test_dice: 0.6615 | test_iou: 0.6203
C01_loss=BCE -> /content/drive/MyDrive/burned_area_outputs_deeplabv3/C01_loss=BCE
  best: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C01_loss=BCE/best.pth
  last: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C01_loss=BCE/last.pth
  hist: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C01_loss=BCE/history.csv
  best_epoch: 30 | test_dice: 0.6485 | test_iou: 0.6094
C02_loss=Focal+Dice -> /content/drive/MyDrive/burned_area_outputs_deeplabv3/C02_loss=Focal+Dice
  best: /content/drive/MyDrive/burned_area_outputs_deeplabv3/C02_loss=F

# **TEST-UNET**

In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================
# STEP 1) TILING + NEGATIVE SAMPLING + EVENT-ID FILENAMES
# ============================================

!pip -q install segmentation_models_pytorch

from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, random
import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

ZIP_ROOT  = "/content/drive/MyDrive/burned_area"
BASE_ROOT = "/content/burned_area"
OUT       = "/content/burned_tiles"

os.makedirs(BASE_ROOT, exist_ok=True)
os.makedirs(f"{OUT}/x", exist_ok=True)
os.makedirs(f"{OUT}/y", exist_ok=True)

# --- unzip all parts ---
for fname in sorted(os.listdir(ZIP_ROOT)):
    if fname.endswith(".zip"):
        part_name = fname.replace(".zip", "")
        out_dir = os.path.join(BASE_ROOT, part_name)
        os.makedirs(out_dir, exist_ok=True)
        print(f"Açılıyor: {fname}")
        with zipfile.ZipFile(os.path.join(ZIP_ROOT, fname), "r") as z:
            z.extractall(out_dir)

TILE = 256

# NEGATIVE SAMPLING SETTINGS
KEEP_NEG_RATIO = 0.15   # 0.10-0.30 arası deneyebilirsin
NEG_PER_POS_CAP = 2.0   # negatifleri pozitiflere göre üstten kıs (örn 2.0 => en fazla 2x negatif)

rng = np.random.default_rng(42)

pos_count = 0
neg_count = 0

# --- iterate PART1..PART5 ---
for part in sorted(os.listdir(BASE_ROOT)):
    part_dir = os.path.join(BASE_ROOT, part)
    if not os.path.isdir(part_dir):
        continue

    # inside: real dataset folder
    subdirs = [d for d in os.listdir(part_dir) if os.path.isdir(os.path.join(part_dir, d))]
    if not subdirs:
        continue
    ROOT = os.path.join(part_dir, subdirs[0])
    print(f"\nİşleniyor: {ROOT}")

    # events
    for event in tqdm(sorted(os.listdir(ROOT))):
        evdir = os.path.join(ROOT, event)
        if not os.path.isdir(evdir):
            continue

        masks = [f for f in os.listdir(evdir) if "mask" in f.lower()]
        s2s = [
            f for f in os.listdir(evdir)
            if f.lower().startswith("sentinel2_")
            and f.lower().endswith(".tiff")
            and "cloud" not in f.lower()
        ]
        if not masks or not s2s:
            continue

        # mask read (assume same H,W as images)
        with rasterio.open(os.path.join(evdir, masks[0])) as msk_src:
            mask_arr = msk_src.read(1)

        for s2 in sorted(s2s):
            with rasterio.open(os.path.join(evdir, s2)) as src:
                if src.count < 3:
                    continue

                H, W = src.height, src.width

                for top in range(0, H - TILE + 1, TILE):
                    for left in range(0, W - TILE + 1, TILE):
                        w = Window(left, top, TILE, TILE)

                        img_patch = src.read([1, 2, 3], window=w).astype(np.float32)  # (3,H,W)
                        mask_patch = mask_arr[top:top+TILE, left:left+TILE]

                        is_pos = (mask_patch > 0).any()

                        # --- balance logic ---
                        if not is_pos:
                            # random downsample negatives
                            if rng.random() > KEEP_NEG_RATIO:
                                continue
                            # cap negatives vs positives
                            if pos_count > 0 and neg_count >= int(NEG_PER_POS_CAP * pos_count):
                                continue

                        # event id & spatial id in filename
                        base_id = f"{part}__{event}__{os.path.splitext(s2)[0]}__t{top}_l{left}"
                        np.save(f"{OUT}/x/{base_id}.npy", img_patch)
                        np.save(f"{OUT}/y/{base_id}.npy", mask_patch)

                        if is_pos:
                            pos_count += 1
                        else:
                            neg_count += 1

print(f"\nSaved patches => pos={pos_count}, neg={neg_count}, total={pos_count+neg_count}")
print("OUT:", OUT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Açılıyor: Satellite_burned_area_dataset_part1.zip
Açılıyor: Satellite_burned_area_dataset_part2.zip
Açılıyor: Satellite_burned_area_dataset_part3.zip
Açılıyor: Satellite_burned_area_dataset_part4.zip
Açılıyor: Satellite_burned_area_dataset_part5.zip

İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part1/Satellite_burned_area_dataset_part1


  0%|          | 0/17 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
100%|██████████| 17/17 [00:15<00:00,  1.08it/s]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part2/Satellite_burned_area_dataset_part2


100%|██████████| 11/11 [00:17<00:00,  1.58s/it]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part3/Satellite_burned_area_dataset_part3


...


İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part4/Satellite_burned_area_dataset_part4


100%|██████████| 7/7 [00:17<00:00,  2.46s/it]



İşleniyor: /content/burned_area/Satellite_burned_area_dataset_part5/Satellite_burned_area_dataset_part5


100%|██████████| 9/9 [00:07<00:00,  1.19it/s]


Saved patches => pos=2330, neg=544, total=2874
OUT: /content/burned_tiles


In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
import os

class BurnedPatchDataset(Dataset):
    def __init__(self, tiles_dir, ids, normalize=True):
        self.tiles_dir = tiles_dir
        self.ids = ids
        self.normalize = normalize

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]

        x = np.load(os.path.join(self.tiles_dir, "x", f"{id_}.npy"))
        y = np.load(os.path.join(self.tiles_dir, "y", f"{id_}.npy"))

        # (H,W,C) gelirse düzelt
        if x.ndim == 3 and x.shape[-1] == 3:
            x = np.transpose(x, (2, 0, 1))

        x = x.astype(np.float32)
        if self.normalize:
            x = x / 255.0

        y = (y > 0).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


In [ ]:
import os
import random
import numpy as np

TILES_DIR = "/content/burned_tiles"
x_dir = os.path.join(TILES_DIR, "x")
y_dir = os.path.join(TILES_DIR, "y")

ids = sorted([os.path.splitext(f)[0] for f in os.listdir(x_dir) if f.endswith(".npy")])

random.seed(42)
random.shuffle(ids)

n = len(ids)
train_ids = ids[:int(0.7*n)]
val_ids   = ids[int(0.7*n):int(0.85*n)]
test_ids  = ids[int(0.85*n):]

print("Train:", len(train_ids), "Val:", len(val_ids), "Test:", len(test_ids))


Train: 2011 Val: 431 Test: 432


In [ ]:
from torch.utils.data import DataLoader
TILES_DIR = "/content/burned_tiles"
train_ds = BurnedPatchDataset(TILES_DIR, train_ids)
val_ds   = BurnedPatchDataset(TILES_DIR, val_ids)
test_ds  = BurnedPatchDataset(TILES_DIR, test_ids)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False)


In [ ]:
import os, glob, re, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_unet"
DATA_ROOT = "/content/burned_tiles"   # <-- eğitimde kullandığın tile dataset root (x/y olan)
print("RUNS_DIR exists:", os.path.exists(RUNS_DIR), RUNS_DIR)
print("DATA_ROOT exists:", os.path.exists(DATA_ROOT), DATA_ROOT)

device: cuda
RUNS_DIR exists: True /content/drive/MyDrive/burned_area_outputs_unet
DATA_ROOT exists: True /content/burned_tiles


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, math
import numpy as np
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_unet"
runs = sorted([d for d in glob.glob(os.path.join(RUNS_DIR, "*")) if os.path.isdir(d)])
print("Found runs:", len(runs))
for r in runs:
    print(" -", os.path.basename(r))

def pick_ckpt(run_dir):
    best = os.path.join(run_dir, "best.pth")
    last = os.path.join(run_dir, "last.pth")
    if os.path.exists(best): return best
    if os.path.exists(last): return last
    # fallback: herhangi bir .pth/.pt bul
    cands = sorted(glob.glob(os.path.join(run_dir, "*.pth")) + glob.glob(os.path.join(run_dir, "*.pt")))
    return cands[0] if len(cands) else None


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
Found runs: 10
 - C00_BASE
 - C01_loss=BCE
 - C02_loss=Focal+Dice
 - C03_sched=plateau
 - C04_amp=on
 - C05_clip=1.0
 - C06_freeze=2ep
 - C07_ES=on
 - C08_lr=5e-5
 - C09_lr=2e-4


In [ ]:
@torch.no_grad()
def dice_iou_from_logits(logits, y, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()
    inter = (pred * y).sum(dim=(1,2,3))

    union_d = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3))
    dice = ((2*inter + eps) / (union_d + eps)).mean().item()

    union_i = pred.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) - inter
    iou = ((inter + eps) / (union_i + eps)).mean().item()
    return dice, iou

@torch.no_grad()
def eval_dice_iou(model, loader, thr=0.5, desc="TEST"):
    model.eval()
    dices, ious = [], []
    for x, y in tqdm(loader, desc=desc):
        x = x.to(device, non_blocking=True)
        y = y.unsqueeze(1).float().to(device, non_blocking=True)
        logits = model(x)
        d, i = dice_iou_from_logits(logits, y, thr=thr)
        dices.append(d); ious.append(i)
    return float(np.mean(dices)), float(np.mean(ious))


# **TEST-İKİ MODEL**

In [ ]:
import os, json, glob
import pandas as pd
import torch
import segmentation_models_pytorch as smp

# ------------------------------------------------------------
# 0) MODEL BUILDER (arch + encoder/weights)
# ------------------------------------------------------------
def build_model_any(arch: str, encoder: str, weights: str):
    arch = arch.lower()
    if arch == "unet":
        return smp.Unet(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
        )
    if arch in ["deeplabv3", "deeplabv3plus", "deeplabv3+"]:
        # Eğitimde DeepLabV3 (plus değil) kullandıysan şunu yap:
        # return smp.DeepLabV3(...)
        return smp.DeepLabV3Plus(
            encoder_name=encoder,
            encoder_weights=weights,
            in_channels=3,
            classes=1,
        )
    raise ValueError(f"Unknown arch={arch}")

# ------------------------------------------------------------
# 1) Run isminden açıklama çıkar (C01_loss=BCE gibi)
# ------------------------------------------------------------
def parse_ablation_from_name(run_name: str):
    if run_name == "C00_BASE":
        return {"ablation_group":"BASE","change_applied":"Baseline config","changed_param":"-","changed_value":"-"}

    tag = run_name.split("_", 1)[-1] if "_" in run_name else run_name

    mapping = {
        "loss=BCE": ("LOSS", "Loss = BCEWithLogits", "loss", "bce"),
        "loss=Focal+Dice": ("LOSS", "Loss = Focal + Dice", "loss", "focal_dice"),
        "sched=plateau": ("SCHED", "Scheduler = ReduceLROnPlateau", "scheduler", "plateau"),
        "amp=on": ("AMP", "AMP = enabled", "amp", True),
        "clip=1.0": ("REG", "Grad clip = 1.0", "grad_clip", 1.0),
        "freeze=2ep": ("FINETUNE", "Encoder frozen for 2 epochs", "freeze_epochs", 2),
        "ES=on": ("ES", "Early stopping = enabled", "early_stopping.enabled", True),
        "lr=5e-5": ("OPT", "LR = 5e-5", "optimizer.lr", 5e-5),
        "lr=2e-4": ("OPT", "LR = 2e-4", "optimizer.lr", 2e-4),
    }
    if tag in mapping:
        g, desc, p, v = mapping[tag]
        return {"ablation_group": g, "change_applied": desc, "changed_param": p, "changed_value": v}

    return {
        "ablation_group": "OTHER",
        "change_applied": f"Custom change: {tag}",
        "changed_param": tag.split("=")[0] if "=" in tag else tag,
        "changed_value": tag.split("=")[1] if "=" in tag else "-"
    }

# ------------------------------------------------------------
# 2) config.json özet
# ------------------------------------------------------------
def cfg_summary(cfg: dict):
    encoder  = cfg.get("encoder", "resnet18")
    weights  = cfg.get("weights", "imagenet")

    loss_cfg = cfg.get("loss", {})
    loss_name = loss_cfg.get("name", "unknown")
    if loss_name == "bce":
        loss_str = "BCE"
    elif loss_name == "bce_dice":
        loss_str = f"BCE+Dice(w_bce={loss_cfg.get('w_bce',1.0)}, w_dice={loss_cfg.get('w_dice',1.0)})"
    elif loss_name == "focal_dice":
        loss_str = f"Focal(a={loss_cfg.get('alpha',0.8)}, g={loss_cfg.get('gamma',2.0)})+Dice"
    else:
        loss_str = str(loss_cfg)

    opt_cfg = cfg.get("optimizer", {})
    opt_name = opt_cfg.get("name", "unknown")
    lr = opt_cfg.get("lr", None)
    wd = opt_cfg.get("weight_decay", None)

    sched_cfg = cfg.get("scheduler", None)
    if sched_cfg is None:
        sched_str = "None"
    else:
        sname = sched_cfg.get("name", "unknown")
        if sname == "plateau":
            sched_str = f"Plateau(factor={sched_cfg.get('factor',0.5)}, patience={sched_cfg.get('patience',2)})"
        elif sname == "cosine":
            sched_str = f"Cosine(T_max={sched_cfg.get('T_max','?')})"
        else:
            sched_str = str(sched_cfg)

    amp = bool(cfg.get("amp", False))
    grad_clip = cfg.get("grad_clip", None)
    freeze_epochs = int(cfg.get("freeze_epochs", 0))
    es_cfg = cfg.get("early_stopping", {})
    es_enabled = bool(es_cfg.get("enabled", False))

    return {
        "encoder": encoder,
        "weights": weights,
        "loss": loss_str,
        "optimizer": opt_name,
        "lr": lr,
        "weight_decay": wd,
        "scheduler": sched_str,
        "amp": amp,
        "grad_clip": grad_clip,
        "freeze_epochs": freeze_epochs,
        "early_stopping": es_enabled
    }

# ------------------------------------------------------------
# 3) RUN klasörlerini bul (RUNS_DIR altındaki tüm run'lar)
# ------------------------------------------------------------
def list_runs(RUNS_DIR: str):
    runs = sorted([d for d in glob.glob(os.path.join(RUNS_DIR, "*")) if os.path.isdir(d)])
    return runs

# ------------------------------------------------------------
# 4) ANA FONKSİYON: belirli arch için test et ve DF döndür
# ------------------------------------------------------------
def test_saved_runs(
    arch: str,
    RUNS_DIR: str,
    device,
    test_loader,
    pick_ckpt_fn,
    eval_fn,
    thr: float = 0.5,
    save_csv: bool = True,
    out_csv_name: str | None = None,
    verbose: bool = True
):
    """
    arch: 'unet' veya 'deeplabv3'
    RUNS_DIR: /content/drive/MyDrive/burned_area_outputs_unet gibi
    pick_ckpt_fn: pick_ckpt fonksiyonun
    eval_fn: eval_dice_iou fonksiyonun
    """

    runs = list_runs(RUNS_DIR)
    if verbose:
        print("ARCH:", arch)
        print("RUNS_DIR:", RUNS_DIR)
        print("Found runs:", len(runs))

    rows = []

    for run_dir in runs:
        name = os.path.basename(run_dir)

        ckpt_path = pick_ckpt_fn(run_dir)
        if ckpt_path is None:
            if verbose: print("[SKIP] no ckpt:", name)
            continue

        # config
        cfg_path = os.path.join(run_dir, "config.json")
        cfg = {}
        if os.path.exists(cfg_path):
            with open(cfg_path, "r") as f:
                cfg = json.load(f)

        summ = cfg_summary(cfg) if cfg else {
            "encoder":"resnet18","weights":"imagenet","loss":"-","optimizer":"-","lr":None,
            "weight_decay":None,"scheduler":"-","amp":None,"grad_clip":None,"freeze_epochs":None,"early_stopping":None
        }

        abl = parse_ablation_from_name(name)

        # ckpt -> state_dict
        ckpt = torch.load(ckpt_path, map_location="cpu")
        sd = ckpt["model"] if (isinstance(ckpt, dict) and "model" in ckpt) else ckpt
        sd = {k.replace("module.", ""): v for k, v in sd.items()}

        # model
        model = build_model_any(arch, summ["encoder"], summ["weights"]).to(device)
        model.load_state_dict(sd, strict=True)
        model.eval()

        # test
        dice, iou = eval_fn(model, test_loader, thr=thr, desc=f"{name} TEST")
        if verbose:
            print(f"[OK] {name} | arch={arch} | {os.path.basename(ckpt_path)} | Dice={dice:.4f} | IoU={iou:.4f}")

        rows.append({
            "run_id": name.split("_")[0] if "_" in name else name,
            "run_tag": name.split("_", 1)[1] if "_" in name else name,

            "arch": arch,
            "ablation_group": abl["ablation_group"],
            "change_applied": abl["change_applied"],
            "changed_param": abl["changed_param"],
            "changed_value": abl["changed_value"],

            "encoder": summ["encoder"],
            "weights": summ["weights"],
            "loss": summ["loss"],
            "optimizer": summ["optimizer"],
            "lr": summ["lr"],
            "weight_decay": summ["weight_decay"],
            "scheduler": summ["scheduler"],
            "amp": summ["amp"],
            "grad_clip": summ["grad_clip"],
            "freeze_epochs": summ["freeze_epochs"],
            "early_stopping": summ["early_stopping"],

            "ckpt_file": os.path.basename(ckpt_path),
            "test_dice@0.5": dice,
            "test_iou@0.5": iou,
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        if verbose:
            print("[WARN] No rows produced. Check RUNS_DIR / ckpt names.")
        return df

    df = df.sort_values("test_iou@0.5", ascending=False).reset_index(drop=True)
    df.insert(0, "rank", range(1, len(df) + 1))

    # save
    if save_csv:
        if out_csv_name is None:
            out_csv_name = f"test_results_{arch}_readable.csv"
        out_csv = os.path.join(RUNS_DIR, out_csv_name)
        df.to_csv(out_csv, index=False)
        if verbose:
            print("\nSaved:", out_csv)

    return df

# ------------------------------------------------------------
# 5) Kullanım örnekleri
# ------------------------------------------------------------
# UNET_RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_unet"
# DLV3_RUNS_DIR = "/content/drive/MyDrive/burned_area_outputs_deeplab"

# df_unet = test_saved_runs("unet", UNET_RUNS_DIR, device, test_loader, pick_ckpt, eval_dice_iou)
# df_dlv3 = test_saved_runs("deeplabv3", DLV3_RUNS_DIR, device, test_loader, pick_ckpt, eval_dice_iou)

# df_unet.head(), df_dlv3.head()


In [ ]:
def make_slide_df(df: pd.DataFrame, mode: str = "medium") -> pd.DataFrame:
    """
    mode:
      - "short"  : en minimal
      - "medium" : slayt için ideal (önerilen)
    """
    # güvenlik: df boşsa direkt dön
    if df is None or len(df) == 0:
        return df

    if mode == "short":
        cols = [
            "rank",
            "run_tag",
            "change_applied",
            "test_iou@0.5",
            "test_dice@0.5",
        ]
    else:  # "medium"
        cols = [
            "rank",
            "run_id",
            "run_tag",
            "change_applied",
            "encoder",
            "loss",
            "lr",
            "test_iou@0.5",
            "test_dice@0.5",
        ]

    # df içinde olmayan sütun varsa drop etmeden seçebilmek için:
    cols = [c for c in cols if c in df.columns]

    out = df[cols].copy()

    # okunurluk: metrikleri kısalt
    if "test_iou@0.5" in out.columns:
        out["test_iou@0.5"] = out["test_iou@0.5"].astype(float).round(3)
    if "test_dice@0.5" in out.columns:
        out["test_dice@0.5"] = out["test_dice@0.5"].astype(float).round(3)
    if "lr" in out.columns:
        # LR'yi bilimsel formatla yazdır
        out["lr"] = out["lr"].apply(lambda x: f"{x:.1e}" if pd.notnull(x) else "-")

    # run_tag uzun ise kısalt
    if "run_tag" in out.columns:
        out["run_tag"] = out["run_tag"].astype(str).str.replace("loss=", "loss:", regex=False)
        out["run_tag"] = out["run_tag"].astype(str).str.replace("sched=", "sched:", regex=False)

    # change_applied uzun ise (slayta sığsın)
    if "change_applied" in out.columns:
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Scheduler = ReduceLROnPlateau", "Scheduler: Plateau", regex=False)
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Loss = BCEWithLogits", "Loss: BCE", regex=False)
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Loss = Focal + Dice", "Loss: Focal+Dice", regex=False)
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Encoder frozen for 2 epochs", "Freeze encoder (2 ep)", regex=False)
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Grad clip = 1.0", "Grad clip: 1.0", regex=False)
        out["change_applied"] = out["change_applied"].astype(str).str.replace("Early stopping = enabled", "Early stopping: ON", regex=False)

    return out


In [ ]:
df_unet = test_saved_runs("unet", "/content/drive/MyDrive/burned_area_outputs_unet",
                          device, test_loader, pick_ckpt, eval_dice_iou)


ARCH: unet
RUNS_DIR: /content/drive/MyDrive/burned_area_outputs_unet
Found runs: 10


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

C00_BASE TEST: 100%|██████████| 54/54 [00:01<00:00, 28.06it/s]


...

[OK] C07_ES=on | arch=unet | best.pth | Dice=0.6212 | IoU=0.5802


C08_lr=5e-5 TEST: 100%|██████████| 54/54 [00:00<00:00, 64.72it/s]


[OK] C08_lr=5e-5 | arch=unet | best.pth | Dice=0.6077 | IoU=0.5639


C09_lr=2e-4 TEST: 100%|██████████| 54/54 [00:00<00:00, 61.38it/s]


[OK] C09_lr=2e-4 | arch=unet | best.pth | Dice=0.6545 | IoU=0.6128

Saved: /content/drive/MyDrive/burned_area_outputs_unet/test_results_unet_readable.csv


In [ ]:
df_unet

,rank,run_id,run_tag,arch,ablation_group,change_applied,changed_param,changed_value,encoder,weights,...,lr,weight_decay,scheduler,amp,grad_clip,freeze_epochs,early_stopping,ckpt_file,test_dice@0.5,test_iou@0.5
0,1,C06,freeze=2ep,unet,FINETUNE,Encoder frozen for 2 epochs,freeze_epochs,2,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,2,False,best.pth,0.652067,0.613453
1,2,C09,lr=2e-4,unet,OPT,LR = 2e-4,optimizer.lr,0.0002,resnet18,imagenet,...,0.00020,0.0001,None,False,NaN,0,False,best.pth,0.654528,0.612849
2,3,C05,clip=1.0,unet,REG,Grad clip = 1.0,grad_clip,1.0,resnet18,imagenet,...,0.00010,0.0001,None,False,1.0,0,False,best.pth,0.653321,0.610096
3,4,C02,loss=Focal+Dice,unet,LOSS,Loss = Focal + Dice,loss,focal_dice,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.648355,0.606606
4,5,C04,amp=on,unet,AMP,AMP = enabled,amp,True,resnet18,imagenet,...,0.00010,0.0001,None,True,NaN,0,False,best.pth,0.644134,0.602910
5,6,C00,BASE,unet,BASE,Baseline config,-,-,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.642110,0.600501
6,7,C01,loss=BCE,unet,LOSS,Loss = BCEWithLogits,loss,bce,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.629697,0.589902
7,8,C03,sched=plateau,unet,SCHED,Scheduler = ReduceLROnPlateau,scheduler,plateau,resnet18,imagenet,...,0.00010,0.0001,"Plateau(factor=0.5, patience=2)",False,NaN,0,False,best.pth,0.624435,0.583495
8,9,C07,ES=on,unet,ES,Early stopping = enabled,early_stopping.enabled,True,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,True,best.pth,0.621173,0.580221
9,10,C08,lr=5e-5,unet,OPT,LR = 5e-5,optimizer.lr,0.00005,resnet18,imagenet,...,0.00005,0.0001,None,False,NaN,0,False,best.pth,0.607662,0.563928


In [ ]:
df_dlv3 = test_saved_runs("deeplabv3", "/content/drive/MyDrive/burned_area_outputs_deeplabv3",
                          device, test_loader, pick_ckpt, eval_dice_iou)

ARCH: deeplabv3
RUNS_DIR: /content/drive/MyDrive/burned_area_outputs_deeplabv3
Found runs: 10


C00_BASE TEST: 100%|██████████| 54/54 [00:01<00:00, 51.16it/s]


[OK] C00_BASE | arch=deeplabv3 | best.pth | Dice=0.6615 | IoU=0.6203


C01_loss=BCE TEST: 100%|██████████| 54/54 [00:00<00:00, 64.03it/s]


[OK] C01_loss=BCE | arch=deeplabv3 | best.pth | Dice=0.6485 | IoU=0.6094


...

[OK] C07_ES=on | arch=deeplabv3 | best.pth | Dice=0.6610 | IoU=0.6212


C08_lr=5e-5 TEST: 100%|██████████| 54/54 [00:00<00:00, 64.31it/s]


[OK] C08_lr=5e-5 | arch=deeplabv3 | best.pth | Dice=0.6352 | IoU=0.5932


C09_lr=2e-4 TEST: 100%|██████████| 54/54 [00:00<00:00, 68.01it/s]


[OK] C09_lr=2e-4 | arch=deeplabv3 | best.pth | Dice=0.6393 | IoU=0.5981

Saved: /content/drive/MyDrive/burned_area_outputs_deeplabv3/test_results_deeplabv3_readable.csv


In [ ]:
df_dlv3

,rank,run_id,run_tag,arch,ablation_group,change_applied,changed_param,changed_value,encoder,weights,...,lr,weight_decay,scheduler,amp,grad_clip,freeze_epochs,early_stopping,ckpt_file,test_dice@0.5,test_iou@0.5
0,1,C05,clip=1.0,deeplabv3,REG,Grad clip = 1.0,grad_clip,1.0,resnet18,imagenet,...,0.00010,0.0001,None,False,1.0,0,False,best.pth,0.695799,0.661692
1,2,C02,loss=Focal+Dice,deeplabv3,LOSS,Loss = Focal + Dice,loss,focal_dice,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.678381,0.634358
2,3,C03,sched=plateau,deeplabv3,SCHED,Scheduler = ReduceLROnPlateau,scheduler,plateau,resnet18,imagenet,...,0.00010,0.0001,"Plateau(factor=0.5, patience=2)",False,NaN,0,False,best.pth,0.666558,0.625595
3,4,C06,freeze=2ep,deeplabv3,FINETUNE,Encoder frozen for 2 epochs,freeze_epochs,2,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,2,False,best.pth,0.663208,0.623197
4,5,C04,amp=on,deeplabv3,AMP,AMP = enabled,amp,True,resnet18,imagenet,...,0.00010,0.0001,None,True,NaN,0,False,best.pth,0.660798,0.621781
5,6,C07,ES=on,deeplabv3,ES,Early stopping = enabled,early_stopping.enabled,True,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,True,best.pth,0.661020,0.621191
6,7,C00,BASE,deeplabv3,BASE,Baseline config,-,-,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.661517,0.620346
7,8,C01,loss=BCE,deeplabv3,LOSS,Loss = BCEWithLogits,loss,bce,resnet18,imagenet,...,0.00010,0.0001,None,False,NaN,0,False,best.pth,0.648450,0.609371
8,9,C09,lr=2e-4,deeplabv3,OPT,LR = 2e-4,optimizer.lr,0.0002,resnet18,imagenet,...,0.00020,0.0001,None,False,NaN,0,False,best.pth,0.639304,0.598097
9,10,C08,lr=5e-5,deeplabv3,OPT,LR = 5e-5,optimizer.lr,0.00005,resnet18,imagenet,...,0.00005,0.0001,None,False,NaN,0,False,best.pth,0.635245,0.593161


In [ ]:
slide_unet = make_slide_df(df_unet, mode="medium")
slide_unet

,rank,run_id,run_tag,change_applied,encoder,loss,lr,test_iou@0.5,test_dice@0.5
0,1,C06,freeze=2ep,Freeze encoder (2 ep),resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.613,0.652
1,2,C09,lr=2e-4,LR = 2e-4,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",2.0e-04,0.613,0.655
2,3,C05,clip=1.0,Grad clip: 1.0,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.610,0.653
3,4,C02,loss:Focal+Dice,Loss: Focal+Dice,resnet18,"Focal(a=0.8, g=2.0)+Dice",1.0e-04,0.607,0.648
4,5,C04,amp=on,AMP = enabled,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.603,0.644
5,6,C00,BASE,Baseline config,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.601,0.642
6,7,C01,loss:BCE,Loss: BCE,resnet18,BCE,1.0e-04,0.590,0.630
7,8,C03,sched:plateau,Scheduler: Plateau,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.583,0.624
8,9,C07,ES=on,Early stopping: ON,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.580,0.621
9,10,C08,lr=5e-5,LR = 5e-5,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",5.0e-05,0.564,0.608


In [ ]:
slide_dlv3 = make_slide_df(df_dlv3, mode="medium")
slide_dlv3

,rank,run_id,run_tag,change_applied,encoder,loss,lr,test_iou@0.5,test_dice@0.5
0,1,C05,clip=1.0,Grad clip: 1.0,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.662,0.696
1,2,C02,loss:Focal+Dice,Loss: Focal+Dice,resnet18,"Focal(a=0.8, g=2.0)+Dice",1.0e-04,0.634,0.678
2,3,C03,sched:plateau,Scheduler: Plateau,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.626,0.667
3,4,C06,freeze=2ep,Freeze encoder (2 ep),resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.623,0.663
4,5,C04,amp=on,AMP = enabled,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.622,0.661
5,6,C07,ES=on,Early stopping: ON,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.621,0.661
6,7,C00,BASE,Baseline config,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",1.0e-04,0.620,0.662
7,8,C01,loss:BCE,Loss: BCE,resnet18,BCE,1.0e-04,0.609,0.648
8,9,C09,lr=2e-4,LR = 2e-4,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",2.0e-04,0.598,0.639
9,10,C08,lr=5e-5,LR = 5e-5,resnet18,"BCE+Dice(w_bce=1.0, w_dice=1.0)",5.0e-05,0.593,0.635


In [ ]:
import pandas as pd
from io import StringIO

raw = """rank	run_id	run_tag	change_applied	encoder	loss	lr	test_iou@0.5	test_dice@0.5
0	1	C06	freeze=2ep	Freeze encoder (2 ep)	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.613	0.652
1	2	C09	lr=2e-4	LR = 2e-4	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	2.0e-04	0.613	0.655
2	3	C05	clip=1.0	Grad clip: 1.0	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.610	0.653
3	4	C02	loss:Focal+Dice	Loss: Focal+Dice	resnet18	Focal(a=0.8, g=2.0)+Dice	1.0e-04	0.607	0.648
4	5	C04	amp=on	AMP = enabled	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.603	0.644
5	6	C00	BASE	Baseline config	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.601	0.642
6	7	C01	loss:BCE	Loss: BCE	resnet18	BCE	1.0e-04	0.590	0.630
7	8	C03	sched:plateau	Scheduler: Plateau	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.583	0.624
8	9	C07	ES=on	Early stopping: ON	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.580	0.621
9	10	C08	lr=5e-5	LR = 5e-5	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	5.0e-05	0.564	0.608
"""

df = pd.read_csv(StringIO(raw), sep="\t")

df_small_unet = df[["rank", "run_id", "change_applied", "test_iou@0.5", "test_dice@0.5"]].copy()

# İstersen sütun isimlerini kısalt:
df_small_unet = df_small_unet.rename(columns={
    "test_iou@0.5": "IoU",
    "test_dice@0.5": "Dice"
})

df_small_unet


,rank,run_id,change_applied,IoU,Dice
0,1,C06,Freeze encoder (2 ep),0.613,0.652
1,2,C09,LR = 2e-4,0.613,0.655
2,3,C05,Grad clip: 1.0,0.610,0.653
3,4,C02,Loss: Focal+Dice,0.607,0.648
4,5,C04,AMP = enabled,0.603,0.644
5,6,C00,Baseline config,0.601,0.642
6,7,C01,Loss: BCE,0.590,0.630
7,8,C03,Scheduler: Plateau,0.583,0.624
8,9,C07,Early stopping: ON,0.580,0.621
9,10,C08,LR = 5e-5,0.564,0.608


In [ ]:
import pandas as pd
from io import StringIO

raw2 = """rank	run_id	run_tag	change_applied	encoder	loss	lr	test_iou@0.5	test_dice@0.5
0	1	C05	clip=1.0	Grad clip: 1.0	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.662	0.696
1	2	C02	loss:Focal+Dice	Loss: Focal+Dice	resnet18	Focal(a=0.8, g=2.0)+Dice	1.0e-04	0.634	0.678
2	3	C03	sched:plateau	Scheduler: Plateau	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.626	0.667
3	4	C06	freeze=2ep	Freeze encoder (2 ep)	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.623	0.663
4	5	C04	amp=on	AMP = enabled	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.622	0.661
5	6	C07	ES=on	Early stopping: ON	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.621	0.661
6	7	C00	BASE	Baseline config	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	1.0e-04	0.620	0.662
7	8	C01	loss:BCE	Loss: BCE	resnet18	BCE	1.0e-04	0.609	0.648
8	9	C09	lr=2e-4	LR = 2e-4	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	2.0e-04	0.598	0.639
9	10	C08	lr=5e-5	LR = 5e-5	resnet18	BCE+Dice(w_bce=1.0, w_dice=1.0)	5.0e-05	0.593	0.635
"""

df2 = pd.read_csv(StringIO(raw2), sep="\t")

df2_small = df2[["rank", "run_id", "change_applied", "test_iou@0.5", "test_dice@0.5"]].copy()

# İstersen sütun isimlerini kısalt:
df2_small = df2_small.rename(columns={
    "test_iou@0.5": "IoU",
    "test_dice@0.5": "Dice"
})

df2_small


,rank,run_id,change_applied,IoU,Dice
0,1,C05,Grad clip: 1.0,0.662,0.696
1,2,C02,Loss: Focal+Dice,0.634,0.678
2,3,C03,Scheduler: Plateau,0.626,0.667
3,4,C06,Freeze encoder (2 ep),0.623,0.663
4,5,C04,AMP = enabled,0.622,0.661
5,6,C07,Early stopping: ON,0.621,0.661
6,7,C00,Baseline config,0.620,0.662
7,8,C01,Loss: BCE,0.609,0.648
8,9,C09,LR = 2e-4,0.598,0.639
9,10,C08,LR = 5e-5,0.593,0.635


# **Model Optimizasyonu**

In [9]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/burned_area_models/runs_burned.zip"
extract_to = "/content/burned_area_models"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

print("Bitti.")
print("Çıkarılan klasör:", extract_to)
print("İçerik:", os.listdir(extract_to))

Bitti.
Çıkarılan klasör: /content/burned_area_models
İçerik: ['content']


In [15]:
zip_path = "/content/drive/MyDrive/burned_area_models/runs_burned_deeplabv3.zip"
extract_to = "/content/burned_area_models"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

print("Bitti.")
print("Çıkarılan klasör:", extract_to)
print("İçerik:", os.listdir(extract_to))

Bitti.
Çıkarılan klasör: /content/burned_area_models
İçerik: ['content']


In [20]:
# ============================================================
# 1) INSTALL
# ============================================================
!pip -q install segmentation-models-pytorch --upgrade

# ============================================================
# 2) IMPORTS
# ============================================================
import os
import copy
import time
import random
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
from tqdm import tqdm

# ============================================================
# 3) GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 4) CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------
TILES_DIR = "/content/burned_tiles"

# ------------------------------------------------------------
# MODEL SELECTION
# sadece bunu değiştir:
# "unet"
# "deeplabv3"
# ------------------------------------------------------------
MODEL_NAME = "unet"

# ------------------------------------------------------------
# CHECKPOINT ROOT
# ------------------------------------------------------------
EXTRACT_ROOT = "/content/burned_area_models/content"

MODEL_RUN_DIRS = {
    "unet": "runs_burned",
    "deeplabv3": "runs_burned_deeplabv3",
    "deeplabv3plus": "runs_burned_deeplabv3",
}

assert MODEL_NAME in MODEL_RUN_DIRS, f"Unsupported MODEL_NAME: {MODEL_NAME}"

RUNS_DIR = os.path.join(EXTRACT_ROOT, MODEL_RUN_DIRS[MODEL_NAME])

# ------------------------------------------------------------
# TRAINING EXPERIMENT SELECTION
# örnekler:
# "C00_BASE"
# "C01_loss=BCE"
# "C02_loss=Focal+Dice"
# "C09_lr=2e-4"
# ------------------------------------------------------------
EXPERIMENT_NAME = "C00_BASE"

CHECKPOINT_PATH = os.path.join(RUNS_DIR, EXPERIMENT_NAME, "best.pth")

ENCODER_NAME = "resnet18"
IN_CHANNELS = 3
CLASSES = 1

# ------------------------------------------------------------
# PRUNING
# ------------------------------------------------------------
PRUNING_TYPE = "unstructured"   # "unstructured" or "structured"
PRUNING_AMOUNT = 0.20
FINETUNE_EPOCHS = 5
FINETUNE_LR = 1e-5

# True -> sparsity-preserving fine-tune
# False -> naive fine-tune
PRESERVE_SPARSITY_DURING_FINETUNE = True

# ------------------------------------------------------------
# LOSS
# checkpoint adına göre otomatik seç
# ------------------------------------------------------------
if "loss=BCE" in EXPERIMENT_NAME:
    LOSS_NAME = "bce"
elif "loss=Focal+Dice" in EXPERIMENT_NAME:
    LOSS_NAME = "focal_dice"
else:
    LOSS_NAME = "bce_dice"

FOCAL_ALPHA = 0.8
FOCAL_GAMMA = 2.0

# ------------------------------------------------------------
# THRESHOLD SEARCH
# ------------------------------------------------------------
THRESHOLDS = np.arange(0.10, 0.91, 0.05)

# ------------------------------------------------------------
# DATALOADER
# ------------------------------------------------------------
BATCH_SIZE = 8
NUM_WORKERS = 2

# ------------------------------------------------------------
# SAVE / EXPORT
# ------------------------------------------------------------
BASE_EXPORT_DIR = "/content/drive/MyDrive/burned_area_models_final"

# Açık deney adı oluştur
PRUNE_TAG = f"prune{int(PRUNING_AMOUNT * 100)}"
FT_TAG = "sparseFT" if PRESERVE_SPARSITY_DURING_FINETUNE else "naiveFT"
FULL_EXPERIMENT_TAG = f"{MODEL_NAME}_{EXPERIMENT_NAME}_{PRUNE_TAG}_{FT_TAG}"

MODEL_EXPORT_DIR = os.path.join(BASE_EXPORT_DIR, MODEL_NAME)
BASELINE_DIR = os.path.join(MODEL_EXPORT_DIR, "baseline")
PRUNING_DIR = os.path.join(MODEL_EXPORT_DIR, "pruning")
REPORTS_DIR = os.path.join(MODEL_EXPORT_DIR, "reports")

os.makedirs(BASELINE_DIR, exist_ok=True)
os.makedirs(PRUNING_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

BASELINE_COPY_PATH = os.path.join(BASELINE_DIR, f"{MODEL_NAME}_{EXPERIMENT_NAME}_best.pth")
SAVE_PRUNED_PATH = os.path.join(PRUNING_DIR, f"{FULL_EXPERIMENT_TAG}.pth")
SAVE_METRICS_CSV = os.path.join(REPORTS_DIR, f"{FULL_EXPERIMENT_TAG}_metrics.csv")
SAVE_SUMMARY_TXT = os.path.join(REPORTS_DIR, f"{FULL_EXPERIMENT_TAG}_summary.txt")

print("MODEL_NAME        :", MODEL_NAME)
print("RUNS_DIR          :", RUNS_DIR)
print("EXPERIMENT_NAME   :", EXPERIMENT_NAME)
print("CHECKPOINT_PATH   :", CHECKPOINT_PATH)
print("BASELINE_COPY     :", BASELINE_COPY_PATH)
print("SAVE_PRUNED_PATH  :", SAVE_PRUNED_PATH)
print("SAVE_METRICS_CSV  :", SAVE_METRICS_CSV)
print("SAVE_SUMMARY_TXT  :", SAVE_SUMMARY_TXT)
print("LOSS_NAME         :", LOSS_NAME)

# ============================================================
# 5) SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ============================================================
# 6) DATASET
# ============================================================
class BurnedPatchDataset(Dataset):
    def __init__(self, tiles_dir, ids, normalize=True):
        self.tiles_dir = tiles_dir
        self.ids = ids
        self.normalize = normalize

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]

        x = np.load(os.path.join(self.tiles_dir, "x", f"{id_}.npy"))
        y = np.load(os.path.join(self.tiles_dir, "y", f"{id_}.npy"))

        if x.ndim == 3 and x.shape[-1] == 3:
            x = np.transpose(x, (2, 0, 1))

        x = x.astype(np.float32)
        if self.normalize:
            x = x / 255.0

        y = (y > 0).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)

# ============================================================
# 7) IDS SPLIT
# ============================================================
x_dir = os.path.join(TILES_DIR, "x")
y_dir = os.path.join(TILES_DIR, "y")

assert os.path.isdir(x_dir), f"x directory not found: {x_dir}"
assert os.path.isdir(y_dir), f"y directory not found: {y_dir}"

ids = sorted([os.path.splitext(f)[0] for f in os.listdir(x_dir) if f.endswith(".npy")])
ids = [id_ for id_ in ids if os.path.exists(os.path.join(y_dir, f"{id_}.npy"))]

random.seed(SEED)
random.shuffle(ids)

n = len(ids)
train_ids = ids[:int(0.7 * n)]
val_ids   = ids[int(0.7 * n):int(0.85 * n)]
test_ids  = ids[int(0.85 * n):]

print("Total :", len(ids))
print("Train :", len(train_ids))
print("Val   :", len(val_ids))
print("Test  :", len(test_ids))

# ============================================================
# 8) DATALOADERS
# ============================================================
train_ds = BurnedPatchDataset(TILES_DIR, train_ids)
val_ds   = BurnedPatchDataset(TILES_DIR, val_ids)
test_ds  = BurnedPatchDataset(TILES_DIR, test_ids)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Train size:", len(train_ds))
print("Val size  :", len(val_ds))
print("Test size :", len(test_ds))

# ============================================================
# 9) MODEL BUILDER
# ============================================================
def build_model():
    if MODEL_NAME == "unet":
        return smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights="imagenet",
            in_channels=IN_CHANNELS,
            classes=CLASSES,
        )

    elif MODEL_NAME == "deeplabv3":
        return smp.DeepLabV3(
            encoder_name=ENCODER_NAME,
            encoder_weights="imagenet",
            in_channels=IN_CHANNELS,
            classes=CLASSES,
            activation=None,
        )

    elif MODEL_NAME == "deeplabv3plus":
        return smp.DeepLabV3Plus(
            encoder_name=ENCODER_NAME,
            encoder_weights="imagenet",
            in_channels=IN_CHANNELS,
            classes=CLASSES,
            activation=None,
        )

    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")

# ============================================================
# 10) CHECKPOINT LOADER
# ============================================================
def load_checkpoint(model, checkpoint_path, device):
    assert os.path.exists(checkpoint_path), f"Checkpoint not found: {checkpoint_path}"

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict):
        if "model" in checkpoint:
            state_dict = checkpoint["model"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    cleaned_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            cleaned_state_dict[k[len("module."):]] = v
        else:
            cleaned_state_dict[k] = v

    model.load_state_dict(cleaned_state_dict, strict=True)
    return model

# baseline checkpoint'i düzenli klasöre kopyala
if not os.path.exists(BASELINE_COPY_PATH):
    shutil.copy2(CHECKPOINT_PATH, BASELINE_COPY_PATH)
    print("Baseline checkpoint copied to:", BASELINE_COPY_PATH)
else:
    print("Baseline checkpoint already exists at:", BASELINE_COPY_PATH)

model = build_model()
model = load_checkpoint(model, CHECKPOINT_PATH, DEVICE)
model = model.to(DEVICE)

print("Checkpoint loaded successfully.")

# ============================================================
# 11) LOSS
# ============================================================
def make_loss(loss_name="bce_dice"):
    if loss_name == "bce":
        bce = torch.nn.BCEWithLogitsLoss()
        return lambda logits, y: bce(logits, y)

    elif loss_name == "bce_dice":
        bce = torch.nn.BCEWithLogitsLoss()
        dice = smp.losses.DiceLoss(mode="binary")
        return lambda logits, y: bce(logits, y) + dice(logits, y)

    elif loss_name == "focal_dice":
        focal = smp.losses.FocalLoss(mode="binary", alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
        dice  = smp.losses.DiceLoss(mode="binary")
        return lambda logits, y: focal(logits, y) + dice(logits, y)

    else:
        raise ValueError(f"Unknown LOSS_NAME: {loss_name}")

criterion = make_loss(LOSS_NAME)

# ============================================================
# 12) METRICS
# ============================================================
@torch.no_grad()
def compute_binary_metrics_from_logits(logits, y, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    pred = (probs > threshold).float()

    pred = pred.view(-1)
    y = y.view(-1)

    tp = (pred * y).sum()
    fp = (pred * (1 - y)).sum()
    fn = ((1 - pred) * y).sum()
    tn = ((1 - pred) * (1 - y)).sum()

    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    dice      = (2 * tp) / (2 * tp + fp + fn + eps)
    iou       = tp / (tp + fp + fn + eps)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + eps)

    return {
        "accuracy": accuracy.item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "dice": dice.item(),
        "iou": iou.item()
    }

# ============================================================
# 13) PRUNING HELPERS
# ============================================================
def apply_unstructured_pruning(model, amount=0.2, verbose=True):
    pruned_layers = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=amount)
            pruned_layers.append(name)

    if verbose:
        print(f"[Unstructured Pruning] amount={amount}")
        print(f"Pruned layer count: {len(pruned_layers)}")
    return model

def apply_structured_pruning(model, amount=0.1, verbose=True):
    pruned_layers = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.out_channels > 8:
            prune.ln_structured(module, name="weight", amount=amount, n=2, dim=0)
            pruned_layers.append(name)

    if verbose:
        print(f"[Structured Pruning] amount={amount}")
        print(f"Pruned layer count: {len(pruned_layers)}")
    return model

def make_pruning_permanent(model):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and hasattr(module, "weight_orig"):
            prune.remove(module, "weight")
    return model

def collect_pruning_masks(model):
    masks = {}
    for name, module in model.named_modules():
        if hasattr(module, "weight_mask"):
            masks[name] = module.weight_mask.detach().clone()
    return masks

def reapply_pruning_masks(model, masks):
    for name, module in model.named_modules():
        if name in masks:
            module.weight.data.mul_(masks[name].to(module.weight.device))

# ============================================================
# 14) TRAIN / EVAL
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, pruning_masks=None):
    model.train()
    running_loss = 0.0

    for x, y in tqdm(loader, desc="Train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.unsqueeze(1).float().to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and device == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        if pruning_masks is not None:
            reapply_pruning_masks(model, pruning_masks)

        running_loss += loss.item()

    return running_loss / len(loader)

@torch.no_grad()
def evaluate_model(model, loader, criterion, device, threshold=0.5, split_name="VAL"):
    model.eval()

    total_loss = 0.0
    metric_sums = {
        "accuracy": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "dice": 0.0,
        "iou": 0.0
    }

    for x, y in tqdm(loader, desc=f"Eval-{split_name}", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.unsqueeze(1).float().to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()

        metrics = compute_binary_metrics_from_logits(logits, y, threshold=threshold)
        for k in metric_sums:
            metric_sums[k] += metrics[k]

    n = len(loader)
    results = {"loss": total_loss / n}
    for k in metric_sums:
        results[k] = metric_sums[k] / n

    return results

@torch.no_grad()
def find_best_threshold(model, loader, device, thresholds=THRESHOLDS):
    model.eval()
    best_threshold = 0.5
    best_dice = -1.0

    for t in thresholds:
        dice_scores = []

        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.unsqueeze(1).float().to(device, non_blocking=True)

            logits = model(x)
            metrics = compute_binary_metrics_from_logits(logits, y, threshold=t)
            dice_scores.append(metrics["dice"])

        avg_dice = float(np.mean(dice_scores))
        if avg_dice > best_dice:
            best_dice = avg_dice
            best_threshold = float(t)

    return best_threshold, best_dice

def finetune_model(model, train_loader, val_loader, criterion, device, epochs=5, lr=1e-5, threshold=0.5, pruning_masks=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

    best_model = copy.deepcopy(model.state_dict())
    best_dice = -1.0
    history = []

    for epoch in range(epochs):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler,
            pruning_masks=pruning_masks
        )

        val_results = evaluate_model(model, val_loader, criterion, device, threshold=threshold, split_name="VAL")

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            **val_results
        })

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_results['loss']:.4f} | "
            f"val_dice={val_results['dice']:.4f} | "
            f"val_iou={val_results['iou']:.4f}"
        )

        if val_results["dice"] > best_dice:
            best_dice = val_results["dice"]
            best_model = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model)

    if pruning_masks is not None:
        reapply_pruning_masks(model, pruning_masks)

    return model, history

# ============================================================
# 15) BENCHMARK (PRUNING-AWARE)
# ============================================================
def get_model_size_mb(model, path="/content/temp_model.pth"):
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    os.remove(path)
    return size_mb

def count_effective_total_and_nonzero_params(model):
    total = 0
    nonzero = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.detach()
            total += w.numel()
            nonzero += (w != 0).sum().item()

            if module.bias is not None:
                b = module.bias.detach()
                total += b.numel()
                nonzero += (b != 0).sum().item()

    return total, nonzero

def get_effective_sparsity(model):
    total, nonzero = count_effective_total_and_nonzero_params(model)
    return 1.0 - (nonzero / total)

def estimate_effective_sparse_size_mb(model):
    _, nonzero = count_effective_total_and_nonzero_params(model)
    return (nonzero * 4) / (1024 * 1024)

def measure_latency(model, input_tensor, device="cuda", warmup=20, runs=100):
    model.eval().to(device)
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)

    if "cuda" in device:
        torch.cuda.synchronize()

    start = time.time()

    with torch.no_grad():
        for _ in range(runs):
            _ = model(input_tensor)

    if "cuda" in device:
        torch.cuda.synchronize()

    end = time.time()
    return (end - start) / runs * 1000.0

def benchmark_model(model, loader, criterion, device, sample_input, threshold=0.5, split_name="VAL"):
    eval_results = evaluate_model(
        model, loader, criterion, device,
        threshold=threshold, split_name=split_name
    )

    runtime_size_mb = get_model_size_mb(model)
    total_params, nonzero_params = count_effective_total_and_nonzero_params(model)
    sparsity = get_effective_sparsity(model)
    effective_sparse_size_mb = estimate_effective_sparse_size_mb(model)
    latency_ms = measure_latency(model, sample_input, device=device)

    return {
        "runtime_size_mb": runtime_size_mb,
        "effective_sparse_size_mb": effective_sparse_size_mb,
        "total_params": total_params,
        "nonzero_params": nonzero_params,
        "sparsity": sparsity,
        "latency_ms": latency_ms,
        **eval_results
    }

# ============================================================
# 16) FULL EXPERIMENT
# ============================================================
def run_pruning_experiment(
    model,
    train_loader,
    val_loader,
    test_loader,
    device,
    sample_input,
    pruning_type="unstructured",
    pruning_amount=0.2,
    finetune_epochs=5,
    finetune_lr=1e-5,
    preserve_sparsity_during_finetune=True,
):
    print("\n================ BASELINE / VAL ================\n")
    baseline_thr, baseline_best_dice = find_best_threshold(model, val_loader, device)
    baseline_val = benchmark_model(model, val_loader, criterion, device, sample_input, threshold=baseline_thr, split_name="VAL")
    baseline_val["best_threshold"] = baseline_thr
    baseline_val["best_val_dice_threshold_search"] = baseline_best_dice
    for k, v in baseline_val.items():
        print(f"{k}: {v}")

    print("\n================ BASELINE / TEST ================\n")
    baseline_test = benchmark_model(model, test_loader, criterion, device, sample_input, threshold=baseline_thr, split_name="TEST")
    baseline_test["used_threshold"] = baseline_thr
    for k, v in baseline_test.items():
        print(f"{k}: {v}")

    pruned_model = copy.deepcopy(model)

    print("\n================ APPLY PRUNING ================\n")
    if pruning_type == "unstructured":
        pruned_model = apply_unstructured_pruning(pruned_model, amount=pruning_amount)
    elif pruning_type == "structured":
        pruned_model = apply_structured_pruning(pruned_model, amount=pruning_amount)
    else:
        raise ValueError("pruning_type must be 'unstructured' or 'structured'")

    pruning_masks = collect_pruning_masks(pruned_model)

    print("\n================ AFTER PRUNING / BEFORE FINETUNE / VAL ================\n")
    pruned_thr, pruned_best_dice = find_best_threshold(pruned_model, val_loader, device)
    pruned_val_before = benchmark_model(pruned_model, val_loader, criterion, device, sample_input, threshold=pruned_thr, split_name="VAL")
    pruned_val_before["best_threshold"] = pruned_thr
    pruned_val_before["best_val_dice_threshold_search"] = pruned_best_dice
    for k, v in pruned_val_before.items():
        print(f"{k}: {v}")

    print("\n================ FINETUNING ================\n")
    active_masks = pruning_masks if preserve_sparsity_during_finetune else None

    pruned_model, history = finetune_model(
        model=pruned_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        device=device,
        epochs=finetune_epochs,
        lr=finetune_lr,
        threshold=pruned_thr,
        pruning_masks=active_masks
    )

    print("\n================ AFTER FINETUNE / VAL ================\n")
    final_thr, final_best_dice = find_best_threshold(pruned_model, val_loader, device)
    final_val = benchmark_model(pruned_model, val_loader, criterion, device, sample_input, threshold=final_thr, split_name="VAL")
    final_val["best_threshold"] = final_thr
    final_val["best_val_dice_threshold_search"] = final_best_dice
    for k, v in final_val.items():
        print(f"{k}: {v}")

    print("\n================ AFTER FINETUNE / TEST ================\n")
    final_test = benchmark_model(pruned_model, test_loader, criterion, device, sample_input, threshold=final_thr, split_name="TEST")
    final_test["used_threshold"] = final_thr
    for k, v in final_test.items():
        print(f"{k}: {v}")

    pruned_model = make_pruning_permanent(pruned_model)

    return {
        "baseline_val": baseline_val,
        "baseline_test": baseline_test,
        "pruned_val_before_ft": pruned_val_before,
        "final_val": final_val,
        "final_test": final_test,
        "history": history,
        "model": pruned_model
    }

# ============================================================
# 17) RUN
# ============================================================
x0, _ = train_ds[0]
C, H, W = x0.shape
sample_input = torch.randn(1, C, H, W)

results = run_pruning_experiment(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=DEVICE,
    sample_input=sample_input,
    pruning_type=PRUNING_TYPE,
    pruning_amount=PRUNING_AMOUNT,
    finetune_epochs=FINETUNE_EPOCHS,
    finetune_lr=FINETUNE_LR,
    preserve_sparsity_during_finetune=PRESERVE_SPARSITY_DURING_FINETUNE,
)

# ============================================================
# 18) SAVE PRUNED MODEL
# ============================================================
save_obj = {
    "model_name": MODEL_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "full_experiment_tag": FULL_EXPERIMENT_TAG,
    "model": results["model"].state_dict(),
    "pruning_type": PRUNING_TYPE,
    "pruning_amount": PRUNING_AMOUNT,
    "loss_name": LOSS_NAME,
    "preserve_sparsity_during_finetune": PRESERVE_SPARSITY_DURING_FINETUNE,
    "history": results["history"],
    "results": {
        "baseline_val": results["baseline_val"],
        "baseline_test": results["baseline_test"],
        "pruned_val_before_ft": results["pruned_val_before_ft"],
        "final_val": results["final_val"],
        "final_test": results["final_test"],
    }
}

torch.save(save_obj, SAVE_PRUNED_PATH)
print(f"\nPruned model saved to: {SAVE_PRUNED_PATH}")

# ============================================================
# 19) SAVE REPORTS
# ============================================================
rows = []
for stage_key in ["baseline_val", "baseline_test", "pruned_val_before_ft", "final_val", "final_test"]:
    d = results[stage_key].copy()
    d["stage"] = stage_key
    d["model_name"] = MODEL_NAME
    d["experiment_name"] = EXPERIMENT_NAME
    d["pruning_type"] = PRUNING_TYPE
    d["pruning_amount"] = PRUNING_AMOUNT
    d["finetune_type"] = FT_TAG
    rows.append(d)

df = pd.DataFrame(rows)
df.to_csv(SAVE_METRICS_CSV, index=False)
print(f"Metrics CSV saved to: {SAVE_METRICS_CSV}")

summary_lines = [
    f"Full Experiment Tag: {FULL_EXPERIMENT_TAG}",
    f"Model Name: {MODEL_NAME}",
    f"Experiment Name: {EXPERIMENT_NAME}",
    f"Checkpoint Path: {CHECKPOINT_PATH}",
    f"Pruning Type: {PRUNING_TYPE}",
    f"Pruning Amount: {PRUNING_AMOUNT}",
    f"Fine-tune Type: {FT_TAG}",
    f"Loss Name: {LOSS_NAME}",
    "",
    "==== FINAL TEST RESULTS ====",
]

for k, v in results["final_test"].items():
    summary_lines.append(f"{k}: {v}")

with open(SAVE_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print(f"Summary TXT saved to: {SAVE_SUMMARY_TXT}")

# ============================================================
# 20) SUMMARY
# ============================================================
def print_summary_block(name, d):
    print("\n" + "="*70)
    print(name)
    print("="*70)
    for k, v in d.items():
        print(f"{k}: {v}")

print_summary_block("BASELINE VAL", results["baseline_val"])
print_summary_block("BASELINE TEST", results["baseline_test"])
print_summary_block("PRUNED BEFORE FT VAL", results["pruned_val_before_ft"])
print_summary_block("FINAL VAL", results["final_val"])
print_summary_block("FINAL TEST", results["final_test"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL_NAME        : unet
RUNS_DIR          : /content/burned_area_models/content/runs_burned
EXPERIMENT_NAME   : C00_BASE
CHECKPOINT_PATH   : /content/burned_area_models/content/runs_burned/C00_BASE/best.pth
BASELINE_COPY     : /content/drive/MyDrive/burned_area_models_final/unet/baseline/unet_C00_BASE_best.pth
SAVE_PRUNED_PATH  : /content/drive/MyDrive/burned_area_models_final/unet/pruning/unet_C00_BASE_prune20_sparseFT.pth
SAVE_METRICS_CSV  : /content/drive/MyDrive/burned_area_models_final/unet/reports/unet_C00_BASE_prune20_sparseFT_metrics.csv
SAVE_SUMMARY_TXT  : /content/drive/MyDrive/burned_area_models_final/unet/reports/unet_C00_BASE_prune20_sparseFT_summary.txt
LOSS_NAME         : bce_dice
Total : 2874
Train : 2011
Val   : 431
Test  : 432
Train size: 2011
Val size  : 431
Test size : 432
Baseline checkpoint copied to: /content/drive/MyDrive/burned_area_

runtime_size_mb: 54.764655113220215
effective_sparse_size_mb: 54.613590240478516
total_params: 14316625
nonzero_params: 14316625
sparsity: 0.0
latency_ms: 4.4361186027526855
loss: 0.3966823273372871
accuracy: 0.9270641130429728
precision: 0.896759698788325
recall: 0.9402901188090995
dice: 0.9140103669078262
iou: 0.849113815360599
best_threshold: 0.7500000000000002
best_val_dice_threshold_search: 0.9140103669078262

================ BASELINE / TEST ================



runtime_size_mb: 54.764655113220215
effective_sparse_size_mb: 54.613590240478516
total_params: 14316625
nonzero_params: 14316625
sparsity: 0.0
latency_ms: 4.5829033851623535
loss: 0.38062928879150637
accuracy: 0.934316529168023
precision: 0.891299135155148
recall: 0.9500413305229611
dice: 0.9140813615587022
iou: 0.8507862157291837
used_threshold: 0.7500000000000002

================ APPLY PRUNING ================

[Unstructured Pruning] amount=0.2
Pruned layer count: 31

================ AFTER PRUNING / BEFORE FINETUNE / VAL ================



runtime_size_mb: 109.38846397399902
effective_sparse_size_mb: 43.69086837768555
total_params: 14316625
nonzero_params: 11453299
sparsity: 0.20000006984886454
latency_ms: 6.008024215698242
loss: 0.3732313189517569
accuracy: 0.930460798519629
precision: 0.9050478129475205
recall: 0.9380292163954841
dice: 0.9175930199799714
iou: 0.8547419740094079
best_threshold: 0.7500000000000002
best_val_dice_threshold_search: 0.9175930199799714

================ FINETUNING ================



Epoch [1/5] | train_loss=0.3267 | val_loss=0.2576 | val_dice=0.9268 | val_iou=0.8693


...

Epoch [3/5] | train_loss=0.2050 | val_loss=0.2371 | val_dice=0.9347 | val_iou=0.8826


Epoch [4/5] | train_loss=0.1735 | val_loss=0.2444 | val_dice=0.9357 | val_iou=0.8843


Epoch [5/5] | train_loss=0.1447 | val_loss=0.2398 | val_dice=0.9372 | val_iou=0.8868

================ AFTER FINETUNE / VAL ================



runtime_size_mb: 109.38846397399902
effective_sparse_size_mb: 43.69086837768555
total_params: 14316625
nonzero_params: 11453299
sparsity: 0.20000006984886454
latency_ms: 5.738961696624756
loss: 0.23978964266953645
accuracy: 0.9473484026061164
precision: 0.9386470350954268
recall: 0.9419778055614896
dice: 0.9381486729339317
iou: 0.8883343272738986
best_threshold: 0.6500000000000001
best_val_dice_threshold_search: 0.9381486729339317

================ AFTER FINETUNE / TEST ================



runtime_size_mb: 109.38846397399902
effective_sparse_size_mb: 43.69086837768555
total_params: 14316625
nonzero_params: 11453299
sparsity: 0.20000006984886454
latency_ms: 5.812361240386963
loss: 0.24860480275970917
accuracy: 0.9480394080833152
precision: 0.9289803957497632
recall: 0.9445183630342837
dice: 0.9339693563955801
iou: 0.8825208290859505
used_threshold: 0.6500000000000001

Pruned model saved to: /content/drive/MyDrive/burned_area_models_final/unet/pruning/unet_C00_BASE_prune20_sparseFT.pth
Metrics CSV saved to: /content/drive/MyDrive/burned_area_models_final/unet/reports/unet_C00_BASE_prune20_sparseFT_metrics.csv
Summary TXT saved to: /content/drive/MyDrive/burned_area_models_final/unet/reports/unet_C00_BASE_prune20_sparseFT_summary.txt

BASELINE VAL
runtime_size_mb: 54.764655113220215
effective_sparse_size_mb: 54.613590240478516
total_params: 14316625
nonzero_params: 14316625
sparsity: 0.0
latency_ms: 4.4361186027526855
loss: 0.3966823273372871
accuracy: 0.9270641130429728
pr